In [1]:
"""
FIFO/LIFO ex-post attribution sensitivity, V8 (2026-11-03)

Inputs: m0.csv; m2-upramp soft0.5.csv; grid_mef_upramp_hourly_2024.csv
Requires numpy and pandas (preinstalled in Colab). No downloads or optimizer.

Six separate virtual labeled pools; zero-carbon initial stocks; opening inventory
withdrawals only; incoming material enters at the end of the hour. Simultaneous
outflow carbon is allocated in proportion to the fixed destination masses. Each
hour contributes one aggregate arrival cohort per pool. All original conversion,
power/battery allocation and cracker-fuel carbon-sink conventions are retained.
These are accounting counterfactuals, not measured physical batch trajectories.
Last-hour labeled cracking is unavailable; sold CI is valid from opening stocks,
but a verified year-end closing stock is NOT reported. No perfect-mixing replay.

V8 diagnoses raw / original-clipped / rescaled / sanitized mass equations before
running. NH3 raw balances close by construction, not by independent validation.
Signed production and sales inside 0.001 kg/h are normalized once for all related
material and direct-carbon terms. Original cracker reconstruction is preserved.
Mass reconciliation preserves all cohort carbon and relative age shares. Its
bound is min(0.010 kg, 0.001 kg + 1e-5 * max(1 kg, saved total physical-product opening mass (both labels))). The
1 g absolute floor reflects the explicit signed-flow budget, not physical mixing
uncertainty. Full source distributions and paired reconciliation-policy runs are
required for interpretation. Limits are engineering audit choices, not inferred
solver feasibility settings. Larger gaps, untraceable carbon and empty-pool
creation fail. Only <=1e-5 kg empty-state dust with <=0.001 kgCO2 can be removed;
its mass and carbon are separately recorded. No fictional cohort is inserted.
All other zero-mass carbon exclusions require an explicit same-hour source match.

Outputs include full hourly/daily/annual ledgers, cohort withdrawal traces,
source-stage diagnostics, per-hour cohort and plant checks, numerical corrections,
and a timestamped ZIP. Input bytes are never edited; SHA256 hashes are recorded.
"""

from collections import deque
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
import zipfile
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION — filename changes only; no modification to model data
# ============================================================
import os
import hashlib
BASE_DIR = Path(os.environ.get('FIFO_LIFO_INPUT_DIR', '/content'))
OUTPUT_DIR = Path(os.environ.get('FIFO_LIFO_OUTPUT_DIR', str(BASE_DIR)))
# Numerical-policy comparison only; default preserves carbon on reconciliation.
RECONCILIATION_POLICY = os.environ.get('FIFO_LIFO_RECON_POLICY', 'preserve_carbon')
if RECONCILIATION_POLICY not in ('preserve_carbon', 'preserve_ci'):
    raise ValueError('Unknown reconciliation policy')
MODEL_FILES = {
    'M0': 'm0.csv',
    'M2-UpRamp MEF0.5': 'm2-upramp soft0.5.csv',
    # Uncomment any extra case YOU HAVE UPLOADED:
    # 'M2-AEF0.5': 'm2-aef0.5.csv',
    # 'M2-Soft MEF0.5': 'm2-soft mef0.5.csv',
}
SIGNAL_FILES = {
    'AEF': ('grid_aef_hourly_2024.csv', 'grid_aef_predicted_kg_per_kwh'),
    'Soft MEF': ('grid_mef_soft_hourly_2024.csv', 'grid_mef_soft_kg_per_kwh'),
    'UpRamp MEF': ('grid_mef_upramp_hourly_2024.csv', 'grid_mef_upramp_pred_kg_per_kwh'),
}
# Default: matched UpRamp comparison only. Add AEF/Soft MEF only if needed.
EVALUATION_SIGNALS = ['UpRamp MEF']
METHODS = ['FIFO', 'LIFO']   # NO perfect-mixing replay in this script
PRODUCTS = ['H2', 'NH3', 'MeOH']
LABELS = ['g', 'ng']

# Constants identical to the original code supplied by the author.
M_H2_TO_NH3 = 0.18
M_H2_TO_MEOH = 0.22
K_H2_OUT = 0.12024
K_NH3_REC = 0.11848
K_H2_FUEL = 0.035294
DE_NH3 = 0.00519      # kgCO2 / kg NH3
DE_MEOH = 0.08        # kgCO2 / kg MeOH
TOL = 1e-6
CRACKER_SPLIT_GAP_KGPH = 5e-4   # EXACT original reconstruction criterion
# A small reconstructed NEGATIVE flow has no meaning as a cohort withdrawal.
# Tolerate only a sub-gram numerical artifact; log every nonzero correction.
RECON_NEGATIVE_EPS_KGPH = 1e-3
# Numerical check bounds, NOT a license to alter physically meaningful flows.
POOL_MASS_ABS_TOL_KG = 1e-2
POOL_MASS_REL_TOL = 1e-7
COHORT_TINY_MASS_KG = 1e-9
COHORT_WITHDRAW_ROUNDOFF_KG = 1e-5  # 10 mg; actual total shortfall <10 mg per real run
# V8: bounded, audited numerical reconciliation, not a license
# to alter solver-fixed production, sales, cracking, or inventory labels.
# This is separate from COHORT_WITHDRAW_ROUNDOFF_KG and carbon tolerances.
MASS_RECON_ABS_LIMIT_KG = 1e-2           # 10 grams
MASS_RECON_REL_LIMIT = 1e-5              # 0.001% of BOTH labels' physical-product opening mass
MASS_RECON_LOG_MIN_KG = 1e-8            # avoid excessive floating-point noise

CARBON_BALANCE_ATOL_KGCO2 = 1e-5
CARBON_BALANCE_RTOL = 1e-9
# V3: maximum EXCLUDED carbon-only grid-power residue per zero-mass cohort.
# Do NOT use this threshold to relax the plant carbon-balance audit.
ZERO_MASS_GRID_CARBON_RESIDUAL_LIMIT_KGCO2 = 1e-3
# V4: universally negligible carbon-only arithmetic roundoff. The 1e-8 bound
# is 100,000 times smaller than the SOURCE-MATCHED 0.001 kgCO2 grid limit.
# It is used ONLY if added material mass <= COHORT_TINY_MASS_KG.
ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2 = 1e-8
# Require carbon-only residue to match the same-hour non-green source-power
# carbon rather than silently discarding inherited/product carbon.
NUMERICAL_SOURCE_MATCH_ATOL_KGCO2 = 1e-8

REQUIRED_MODEL_COLS = [
    'model_tag', 'datetime_utc', 'day_of_year', 'hour_in_day',
    'P_elz_mw', 'P_nh3_mw', 'P_meoh_mw', 'P_crk_mw',
    'elz_power_green_share', 'nh3_power_green_share',
    'meoh_power_green_share', 'crk_power_green_share',
    'grid_import_mw', 'res_used_mw', 'bat_ch_mw', 'bat_dis_mw',
    'H2_prod_g_kgph', 'H2_prod_ng_kgph',
    'NH3_prod_g_kgph', 'NH3_prod_ng_kgph',
    'MeOH_prod_g_kgph', 'MeOH_prod_ng_kgph',
    'crk_nh3_in_kgph', 'crk_h2_out_kgph', 'crk_h2_fuel_kgph',
    'H2_sell_g_kgph', 'H2_sell_ng_kgph',
    'NH3_sell_g_kgph', 'NH3_sell_ng_kgph',
    'MeOH_sell_g_kgph', 'MeOH_sell_ng_kgph',
    'tank_H2_g_kg', 'tank_H2_ng_kg',
    'tank_NH3_g_kg', 'tank_NH3_ng_kg',
    'tank_MeOH_g_kg', 'tank_MeOH_ng_kg',
]
POOL_NAMES = [f'{p}_{label}' for p in PRODUCTS for label in LABELS]


def mass_tolerance(mass):
    return max(POOL_MASS_ABS_TOL_KG, POOL_MASS_REL_TOL * max(1.0, abs(mass)))


def check_close_mass(actual, expected, context):
    gap = actual - expected
    if abs(gap) > mass_tolerance(expected):
        raise ValueError(f'MATERIAL BALANCE FAILURE: {context}: '
                         f'cohorts={actual:.9f}, optimizer={expected:.9f}, gap={gap:.9f} kg')
    return gap


def assert_nonnegative(value, context, tolerance=RECON_NEGATIVE_EPS_KGPH):
    if not np.isfinite(value):
        raise ValueError(f'Nonfinite mass in {context}: {value}')
    if value < -tolerance:
        raise ValueError(f'Negative material flow in {context}: {value} kg')
    return float(max(value, 0.0))


# ============================================================
# ORIGINAL INPUT AND ELECTRICITY RECONSTRUCTION — kept unchanged
# ============================================================
def clip_small_negative(series, tol=TOL):
    x = series.copy()
    x[(x < 0) & (x > -tol)] = 0.0
    return x


def load_model_df(path):
    if not path.exists():
        raise FileNotFoundError(f'Missing model file: {path}')
    d = pd.read_csv(path)
    missing = [c for c in REQUIRED_MODEL_COLS if c not in d.columns]
    if missing:
        raise KeyError(f'{path.name} missing required columns: {missing}')
    d = d.copy()
    d['datetime_utc'] = pd.to_datetime(d['datetime_utc'], utc=True)
    d = d.sort_values('datetime_utc').reset_index(drop=True)
    if d['datetime_utc'].isna().any() or d['datetime_utc'].duplicated().any():
        raise ValueError(f'{path.name}: missing or duplicate hourly timestamps')
    # Complete 2024 studies; no silent gaps or misplaced days.
    expected = pd.date_range('2024-01-01', periods=8784, freq='h', tz='UTC')
    if not pd.DatetimeIndex(d['datetime_utc']).equals(expected):
        raise ValueError(f'{path.name}: expected exact 8784 hourly UTC observations for 2024')
    # Original optimizer convention: when a processing unit is OFF, its green
    # power share is undefined (NaN), because the export calculates g/(g+ng)
    # only for nonzero total unit power. The author's ORIGINAL CI reconstruction
    # explicitly converts such NaNs to zero using fillna(0.0). They must NOT
    # cause input rejection. All other required numeric source fields must be
    # present: these fix physical production, withdrawals, and inventory states.
    share_units = ('elz', 'nh3', 'meoh', 'crk')
    share_cols = [f'{unit}_power_green_share' for unit in share_units]
    physical_cols = [c for c in REQUIRED_MODEL_COLS[4:] if c not in share_cols]
    for col in physical_cols:
        d[col] = pd.to_numeric(d[col], errors='coerce')
        bad = ~np.isfinite(d[col].to_numpy(dtype=float))
        if bad.any():
            examples = d.loc[bad, 'datetime_utc'].head(3).astype(str).tolist()
            raise ValueError(f'{path.name}: nonfinite/missing required field {col} '
                             f'({int(bad.sum())} rows; example hours: {examples})')
    for unit in share_units:
        col = f'{unit}_power_green_share'
        d[col] = pd.to_numeric(d[col], errors='coerce')
        # A missing share is scientifically meaningful ONLY when its unit is off.
        # A missing share during positive production is ambiguous and would
        # silently misattribute grid electricity if replaced with zero.
        bad = d[col].isna() & (d[f'P_{unit}_mw'] > TOL)
        if bad.any():
            examples = d.loc[bad, 'datetime_utc'].head(3).astype(str).tolist()
            raise ValueError(f'{path.name}: missing {col} while unit power is '
                             f'positive ({int(bad.sum())} rows; hours: {examples})')
        invalid=d[col].notna() & (~np.isfinite(d[col]) | (((d[col] < -TOL) | (d[col] > 1+TOL)) & (d[f'P_{unit}_mw']>TOL)))
        if invalid.any():
            raise ValueError(f'{path.name}: invalid power share {col}')
        # Leave legitimate idle-hour NaNs intact. The UNCHANGED original
        # reconstruct_unit_grid_battery_split() clips and fills these with 0.
    return d


def load_signal_df(path, ef_col):
    if not path.exists():
        raise FileNotFoundError(f'Missing emission signal: {path}')
    d = pd.read_csv(path)
    if 'datetime_utc' not in d or ef_col not in d:
        raise KeyError(f'{path.name} must include datetime_utc and {ef_col}')
    d = d[['datetime_utc', ef_col]].copy()
    d['datetime_utc'] = pd.to_datetime(d['datetime_utc'], utc=True)
    if d['datetime_utc'].duplicated().any():
        raise ValueError(f'{path.name}: duplicate emission-signal timestamps')
    d = d.sort_values('datetime_utc').rename(columns={ef_col: 'ef_kg_per_kwh'})
    if (~np.isfinite(pd.to_numeric(d['ef_kg_per_kwh'],errors='coerce'))).any():
        raise ValueError(f'{path.name}: missing emission-signal values')
    if (d['ef_kg_per_kwh'] < 0).any():
        raise ValueError('Negative emission factor unsupported by nonnegative carbon cohorts')
    return d


def reconstruct_unit_grid_battery_split(df, tol=TOL):
    # Exact logic of the author's original script; no new electricity assumption.
    d = df.copy()
    for unit in ('elz', 'nh3', 'meoh', 'crk'):
        d[f'P_{unit}_total_mw'] = d[f'P_{unit}_mw']
        share = f'{unit}_power_green_share'
        d[share] = pd.to_numeric(d[share], errors='coerce').clip(0.0, 1.0).fillna(0.0)
        d[f'P_{unit}_green_mw'] = d[f'P_{unit}_total_mw'] * d[share]
        d[f'P_{unit}_ng_mw'] = clip_small_negative(d[f'P_{unit}_total_mw'] - d[f'P_{unit}_green_mw'], tol)
    d['P_ng_sum_units_mw'] = sum(d[f'P_{u}_ng_mw'] for u in ('elz', 'nh3', 'meoh', 'crk'))
    for unit in ('elz', 'nh3', 'meoh', 'crk'):
        unit_ng = d[f'P_{unit}_ng_mw']
        d[f'P_{unit}_bat_mw'] = np.where(
            (d['P_ng_sum_units_mw'] > tol) & (d['bat_dis_mw'] > tol),
            d['bat_dis_mw'] * unit_ng / d['P_ng_sum_units_mw'].replace(0.0, np.nan), 0.0,
        )
        d[f'P_{unit}_grid_exact_mw'] = clip_small_negative(
            d[f'P_{unit}_ng_mw'] - d[f'P_{unit}_bat_mw'], tol)
    d['grid_unit_sum_exact_mw'] = sum(d[f'P_{u}_grid_exact_mw'] for u in ('elz', 'nh3', 'meoh', 'crk'))
    d['bat_unit_sum_exact_mw'] = sum(d[f'P_{u}_bat_mw'] for u in ('elz', 'nh3', 'meoh', 'crk'))
    d['grid_balance_gap_mw'] = d['grid_unit_sum_exact_mw'] - d['grid_import_mw']
    d['bat_balance_gap_mw'] = d['bat_unit_sum_exact_mw'] - d['bat_dis_mw']
    diag = pd.DataFrame([
        {'diagnostic': f'{kind}_{agg}_gap_mw', 'value': float(v)}
        for kind, series in [('grid', d['grid_balance_gap_mw']), ('battery', d['bat_balance_gap_mw'])]
        for agg, v in [('max_abs', series.abs().max()), ('mean_abs', series.abs().mean())]
    ])
    return d, diag


def reconstruct_intermediates(df):
    # Matches the author's archived algorithm, including last-hour validity flag.
    d = df.copy()
    for p in PRODUCTS:
        for label in LABELS:
            opening = f'tank_{p}_{label}_kg'
            d[f'tank_{p}_{label}_next_kg'] = d[opening].shift(-1)
            d[f'delta_tank_{p}_{label}_kg'] = d[f'tank_{p}_{label}_next_kg'] - d[opening]
    d['has_next_state'] = d['tank_H2_g_next_kg'].notna()
    for label in LABELS:
        d[f'H_to_NH3_{label}_kgph'] = M_H2_TO_NH3 * d[f'NH3_prod_{label}_kgph']
        d[f'H_to_MeOH_{label}_kgph'] = M_H2_TO_MEOH * d[f'MeOH_prod_{label}_kgph']
    denom = 1.0 - K_NH3_REC
    for label in LABELS:
        d[f'Q_crk_in_{label}_recon_kgph'] = (
            d[f'NH3_prod_{label}_kgph'] - d[f'NH3_sell_{label}_kgph']
            - d[f'delta_tank_NH3_{label}_kg']
        ) / denom
        d[f'Q_crk_in_{label}_recon_kgph'] = clip_small_negative(d[f'Q_crk_in_{label}_recon_kgph'])
    d['Q_crk_in_recon_sum_kgph'] = d['Q_crk_in_g_recon_kgph'] + d['Q_crk_in_ng_recon_kgph']
    d['Q_crk_total_gap_kgph'] = d['Q_crk_in_recon_sum_kgph'] - d['crk_nh3_in_kgph']
    d['Q_crk_split_valid'] = (
        d['has_next_state'] & d['Q_crk_in_g_recon_kgph'].notna()
        & d['Q_crk_in_ng_recon_kgph'].notna()
        & (d['Q_crk_total_gap_kgph'].abs() <= CRACKER_SPLIT_GAP_KGPH)
    )
    valid = d['Q_crk_split_valid'] & (d['Q_crk_in_recon_sum_kgph'] > TOL)
    scale = np.where(valid, d['crk_nh3_in_kgph'] / d['Q_crk_in_recon_sum_kgph'].replace(0, np.nan), 1.0)
    for label in LABELS:
        d.loc[valid, f'Q_crk_in_{label}_recon_kgph'] *= scale[valid]
        d[f'NH3_rec_{label}_recon_kgph'] = K_NH3_REC * d[f'Q_crk_in_{label}_recon_kgph']
        d[f'H_crk_out_{label}_recon_kgph'] = K_H2_OUT * d[f'Q_crk_in_{label}_recon_kgph']
        d[f'H_crk_fuel_{label}_recon_kgph'] = K_H2_FUEL * d[f'Q_crk_in_{label}_recon_kgph']
    d['H_crk_out_total_gap_kgph'] = (
        d['H_crk_out_g_recon_kgph'] + d['H_crk_out_ng_recon_kgph'] - d['crk_h2_out_kgph'])
    d['H_crk_fuel_total_gap_kgph'] = (
        d['H_crk_fuel_g_recon_kgph'] + d['H_crk_fuel_ng_recon_kgph'] - d['crk_h2_fuel_kgph'])
    diag = pd.DataFrame([
        {'diagnostic': 'n_rows_total', 'value': len(d)},
        {'diagnostic': 'n_valid_cracker_split', 'value': int(d['Q_crk_split_valid'].sum())},
        {'diagnostic': 'n_unavailable_cracker_split', 'value': int((~d['Q_crk_split_valid']).sum())},
        {'diagnostic': 'max_abs_cracker_nh3_in_gap_kgph', 'value': d['Q_crk_total_gap_kgph'].abs().max()},
        {'diagnostic': 'max_abs_cracker_h2_out_gap_kgph', 'value': d['H_crk_out_total_gap_kgph'].abs().max()},
        {'diagnostic': 'max_abs_cracker_h2_fuel_gap_kgph', 'value': d['H_crk_fuel_total_gap_kgph'].abs().max()},
    ])
    # Fail if a missing internal feed split would otherwise silently discard material.
    interior_bad = (~d['Q_crk_split_valid']) & d['has_next_state'] & (d['crk_nh3_in_kgph'].abs() > TOL)
    if interior_bad.any():
        row = d.loc[interior_bad].iloc[0]
        raise ValueError(f'Nonfinal hour has unresolvable labeled cracker feed at {row.datetime_utc}; '
                         'cannot safely perform FIFO/LIFO accounting')
    return d, diag


# ============================================================
# MATERIAL-BALANCE PREFLIGHT — independent of accounting method/signal
# ============================================================
def material_source_balance_audit(recon_df):
    """Verify every saved next-hour labeled inventory against fixed flows.

    This does NOT recalculate carbon or edit the original CSV. It uses the
    numerical nonnegative-flow convention of run_fifo_lifo_ledger so that
    the material preflight and actual ledger compare like with like.
    """
    d = recon_df
    for p in PRODUCTS:
        for lab in LABELS:
            for kind in ('prod', 'sell'):
                raw = d[f'{p}_{kind}_{lab}_kgph'].to_numpy(dtype=float)
                if not np.isfinite(raw).all() or np.any(raw < -RECON_NEGATIVE_EPS_KGPH):
                    raise ValueError(f'Invalid {p}_{kind}_{lab} source flow in mass preflight')
    rows = []
    for i in range(len(d)-1):
        r, nxt = d.iloc[i], d.iloc[i+1]
        ts = r['datetime_utc']
        for lab in LABELS:
            q = assert_nonnegative(float(r[f'Q_crk_in_{lab}_recon_kgph']),
                                   f'preflight cracker feed {lab} at {ts}')
            h2_to_nh3 = assert_nonnegative(float(r[f'H_to_NH3_{lab}_kgph']),
                                         f'preflight H2 to NH3 {lab} at {ts}')
            h2_to_meoh = assert_nonnegative(float(r[f'H_to_MeOH_{lab}_kgph']),
                                          f'preflight H2 to MeOH {lab} at {ts}')
            for p in PRODUCTS:
                production = assert_nonnegative(float(r[f'{p}_prod_{lab}_kgph']),
                                                f'preflight {p} production {lab} at {ts}')
                sale = assert_nonnegative(float(r[f'{p}_sell_{lab}_kgph']),
                                          f'preflight {p} sale {lab} at {ts}')
                old = float(r[f'tank_{p}_{lab}_kg'])
                actual_next = float(nxt[f'tank_{p}_{lab}_kg'])
                if p == 'H2':
                    predicted = old + production + K_H2_OUT*q - sale - h2_to_nh3 - h2_to_meoh - K_H2_FUEL*q
                elif p == 'NH3':
                    predicted = old + production + K_NH3_REC*q - sale - q
                else:
                    predicted = old + production - sale
                gap = actual_next - predicted
                bound_relative = RECON_NEGATIVE_EPS_KGPH + MASS_RECON_REL_LIMIT * max(1.0, sum(abs(float(nxt[f'tank_{p}_{x}_kg'])) for x in LABELS))
                safe = abs(gap) <= MASS_RECON_ABS_LIMIT_KG and abs(gap) <= bound_relative
                rows.append({
                    'datetime_utc': ts, 'pool': f'{p}_{lab}',
                    'solver_opening_kg': old,
                    'next_solver_opening_kg': actual_next,
                    'predicted_next_from_source_flows_kg': predicted,
                    'raw_source_balance_gap_kg': gap,
                    'abs_gap_limit_kg': MASS_RECON_ABS_LIMIT_KG,
                    'relative_gap_limit_kg': bound_relative,
                    'physical_product_next_mass_kg':sum(abs(float(nxt[f'tank_{p}_{x}_kg'])) for x in LABELS),
                    'within_reconciliation_bounds': bool(safe),
                })
    result = pd.DataFrame(rows)
    bad = result.loc[~result['within_reconciliation_bounds']]
    if len(bad):
        sample = bad.iloc[0]
        raise ValueError(
            f'REAL SOURCE MATERIAL-BALANCE GAP too large in preflight: '
            f'{sample.pool} at {sample.datetime_utc}, gap={sample.raw_source_balance_gap_kg:.12g} kg, '
            f'absolute limit={MASS_RECON_ABS_LIMIT_KG} kg, '
            f'relative limit={sample.relative_gap_limit_kg:.12g} kg; '
            f'{len(bad)} of {len(result)} source balances failed. '
            'No FIFO/LIFO results will be reported.')
    return result


# ============================================================
# COHORT INVENTORY — ONLY NEW SCIENTIFIC STATE / ATTRIBUTION RULE
# ============================================================
@dataclass
class Cohort:
    timestamp: object
    mass_kg: float
    carbon_kgco2: float


class CohortPool:
    """Chronological carbon cohorts. Timestamp reflects entry hour, NOT physical batch ID."""
    def __init__(self, name, mode, initial_mass):
        if mode not in ('FIFO', 'LIFO'):
            raise ValueError('Only FIFO/LIFO are implemented')
        self.name, self.mode = name, mode
        self.q = deque()
        self.mass_kg = 0.0
        self.carbon_kgco2 = 0.0
        self.roundoff_kg = 0.0
        self.roundoff_count = 0
        self.reconciliation_carbon_change = 0.0
        self.dust_mass_removed = 0.0
        self.suppressed_numerical_carbon_kgco2 = 0.0
        self.suppressed_numerical_carbon_events = 0
        self.add(initial_mass, 0.0, 'initial_inventory_before_2024')

    def add(self, mass, carbon, stamp, source_grid_carbon_residual=None):
        """Add a physical material cohort; return numerical carbon excluded (kgCO2).

        The V4/V5/V6 checks distinguish vanishingly small arithmetic roundoff
        (<=1e-8 kgCO2 in any pool, including green) from larger, provably
        source-matched non-green grid-power residuals (<=0.001 kgCO2).
        No substantial or untraceable carbon is discarded. Both are
        explicit sinks in the plant carbon conservation audit.
        """
        mass = assert_nonnegative(float(mass), f'{self.name} addition mass at {stamp}')
        if not np.isfinite(carbon) or carbon < 0.0:
            raise ValueError(f'Invalid added carbon in {self.name} at {stamp}: {carbon}')
        carbon = float(max(0.0, carbon))
        if mass <= COHORT_TINY_MASS_KG:
            if carbon > 0.0:
                if carbon > ZERO_MASS_GRID_CARBON_RESIDUAL_LIMIT_KGCO2:
                    raise ValueError(
                        f'Carbon without mass in {self.name} at {stamp}: {carbon} kgCO2; '
                        f'exceeds zero-mass grid-residual limit '
                        f'{ZERO_MASS_GRID_CARBON_RESIDUAL_LIMIT_KGCO2} kgCO2')
                # V4: <=1e-8 kgCO2 is sub-microgram-scale arithmetic
                # roundoff; it can originate from inherited/partitioned
                # carbon in a GREEN pool without any non-green grid source.
                # Do not assert a source that we cannot verify.
                # Above that narrow bound, require the original V3
                # non-green source-grid match and <=0.001 kgCO2.
                if carbon > ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2:
                    if source_grid_carbon_residual is None or not np.isfinite(source_grid_carbon_residual):
                        raise ValueError(
                            f'Carbon without mass in {self.name} at {stamp}: {carbon} kgCO2; '
                            'no matching non-green process-grid source')
                    if abs(carbon - source_grid_carbon_residual) > NUMERICAL_SOURCE_MATCH_ATOL_KGCO2:
                        raise ValueError(
                            f'Carbon without mass in {self.name} at {stamp}: {carbon} kgCO2; '
                            f'does not match source-grid residual {source_grid_carbon_residual} kgCO2')
                self.suppressed_numerical_carbon_kgco2 += carbon
                self.suppressed_numerical_carbon_events += 1
            return carbon
        # Every hour contributes at most one aggregated cohort per labeled pool.
        self.q.append(Cohort(stamp, mass, carbon))
        self.mass_kg += mass
        self.carbon_kgco2 += carbon
        return 0.0

    def withdraw(self, requested_mass, stamp, trace=None):
        requested_mass = assert_nonnegative(float(requested_mass), f'{self.name} withdrawal at {stamp}')
        if requested_mass <= COHORT_TINY_MASS_KG:
            return 0.0
        if requested_mass > self.mass_kg + COHORT_WITHDRAW_ROUNDOFF_KG:
            raise ValueError(f'Withdrawal exceeds opening material pool {self.name} at {stamp}: '
                             f'request={requested_mass}, available={self.mass_kg}')
        if requested_mass > self.mass_kg:
            self.roundoff_kg += requested_mass - self.mass_kg
            self.roundoff_count += 1
        remaining = min(requested_mass, self.mass_kg)
        carbon_withdrawn = 0.0
        while remaining > COHORT_TINY_MASS_KG:
            if not self.q:
                raise ValueError(f'Empty cohort queue during {self.name} withdrawal at {stamp}')
            cohort = self.q.popleft() if self.mode == 'FIFO' else self.q.pop()
            take = min(remaining, cohort.mass_kg)
            # Proportionally deplete each cohort's carbon; its individual CI is fixed.
            take_carbon = cohort.carbon_kgco2 * (take / cohort.mass_kg)
            cohort.mass_kg -= take
            cohort.carbon_kgco2 -= take_carbon
            if trace is not None:
                trace.append({
                    'datetime_utc': stamp, 'source_pool': self.name,
                    'cohort_origin_hour': str(cohort.timestamp),
                    'withdrawn_cohort_mass_kg': take,
                    'withdrawn_cohort_carbon_kgco2': take_carbon,
                })
            remaining -= take
            carbon_withdrawn += take_carbon
            self.mass_kg -= take
            self.carbon_kgco2 -= take_carbon
            if cohort.mass_kg > COHORT_TINY_MASS_KG:
                if self.mode == 'FIFO':
                    self.q.appendleft(cohort)
                else:
                    self.q.append(cohort)
            elif abs(cohort.carbon_kgco2) > CARBON_BALANCE_ATOL_KGCO2:
                raise ValueError(f'Unassigned residual carbon when cohort exhausted in {self.name}')
        if abs(self.mass_kg) < COHORT_TINY_MASS_KG:
            self.mass_kg = 0.0
        # V5: DO NOT zero a tiny running carbon stock independently of its
        # cohorts. The sum of carbon in individual remaining cohorts is the
        # authoritative carbon stock, even when its magnitude is <1e-5 kgCO2.
        # V4 set only self.carbon_kgco2 to zero here while retaining the
        # carbon in self.q; repeating this created an artificial mismatch.
        # Any roundoff in an exhausted cohort is instead checked explicitly
        # above, and internal totals are independently audited below.
        return float(carbon_withdrawn)

    def reconcile_solver_opening(self, target_mass, stamp, physical_tank_mass=None):
        """Reconcile numerical cohort-mass drift WITHOUT changing carbon.

        The optimizer's saved total physical-product opening mass (both labels) is authoritative. A bounded
        discrepancy is spread across existing cohort masses proportionally,
        preserving chronological order, positive cohort shares and aggregate
        carbon. This adjustment is ONLY accounting-numerical; no solver flow
        or process carbon generation is changed. Distinct carbon values are
        retained, so cohort CI changes very slightly with corrected mass.

        Return raw signed mass gap (target minus cohort), or raise on an
        unexplained / large gap. Empty pool repair is not invented.
        """
        self.reconciliation_carbon_change = 0.0
        self.dust_mass_removed = 0.0
        target = assert_nonnegative(float(target_mass), f'{self.name} solver opening at {stamp}')
        before = self.mass_kg
        delta = target - before
        if abs(delta) <= MASS_RECON_LOG_MIN_KG:
            return delta
        bound_relative = RECON_NEGATIVE_EPS_KGPH + MASS_RECON_REL_LIMIT * max(1.0, target if physical_tank_mass is None else physical_tank_mass)
        if abs(delta) > MASS_RECON_ABS_LIMIT_KG or abs(delta) > bound_relative:
            raise ValueError(
                f'UNRESOLVED solver/cohort mass discrepancy in {self.name} at {stamp}: '
                f'solver_opening={target:.12g} kg, cohort_opening={before:.12g} kg, '
                f'delta={delta:.12g} kg; limits: {MASS_RECON_ABS_LIMIT_KG} kg '
                f'AND {bound_relative:.12g} kg floor-plus-relative. Inspect material-balance audit.')
        # Do not invent carbon-free cohorts for a zero/empty inventory.
        if before <= COHORT_TINY_MASS_KG or target <= COHORT_TINY_MASS_KG or not self.q:
            # Explicit empty-state dust exclusion, with carbon in plant audit.
            # Never create a cohort when the source shows a tiny positive stock.
            if max(before, target) <= 1e-5 and abs(self.carbon_kgco2) <= ZERO_MASS_GRID_CARBON_RESIDUAL_LIMIT_KGCO2:
                self.reconciliation_carbon_change = -self.carbon_kgco2
                self.dust_mass_removed = before
                self.q.clear()
                self.mass_kg = self.carbon_kgco2 = 0.0
                return -before
            raise ValueError(
                f'Cannot safely reconcile empty or nearly empty {self.name} pool at {stamp}: '
                f'solver={target:.12g} kg, cohorts={before:.12g} kg, '
                f'carbon_stock={self.carbon_kgco2:.12g} kgCO2')
        factor = target / before
        for item in self.q:
            item.mass_kg *= factor
            if RECONCILIATION_POLICY == 'preserve_ci':
                item.carbon_kgco2 *= factor
        if RECONCILIATION_POLICY == 'preserve_ci':
            after_carbon = sum(item.carbon_kgco2 for item in self.q)
            self.reconciliation_carbon_change = after_carbon - self.carbon_kgco2
            self.carbon_kgco2 = after_carbon
        # Correct just the last arithmetic summation residue, without
        # targeting oldest/newest cohort to absorb the real solver gap.
        summed = sum(item.mass_kg for item in self.q)
        rounding = target - summed
        if rounding:
            largest = max(self.q, key=lambda b: b.mass_kg)
            if largest.mass_kg + rounding <= 0.0:
                raise ValueError('Numerical mass closure produced nonpositive cohort')
            largest.mass_kg += rounding
        self.mass_kg = target
        # Default policy preserves carbon; preserve_ci is only a numerical sensitivity.
        self.verify_internal_sums(stamp)
        return delta

    def verify_internal_sums(self, stamp):
        mass = sum(b.mass_kg for b in self.q)
        carbon = sum(b.carbon_kgco2 for b in self.q)
        if abs(mass-self.mass_kg) > 1e-7 + 1e-10*max(1.0, abs(mass)):
            raise ValueError(f'Internal cohort mass mismatch {self.name} at {stamp}: {mass-self.mass_kg}')
        if any(b.mass_kg < 0 or b.carbon_kgco2 < -1e-10 for b in self.q):
            raise ValueError(f'Negative cohort state {self.name} at {stamp}')
        carbon_gap = carbon - self.carbon_kgco2
        if abs(carbon_gap) > max(CARBON_BALANCE_ATOL_KGCO2,
                                 CARBON_BALANCE_RTOL * max(1.0, abs(carbon), abs(self.carbon_kgco2))):
            raise ValueError(
                f'Cohort carbon sum mismatch in {self.name} at {stamp}: '
                f'sum_of_cohorts={carbon:.12g} kgCO2, '
                f'running_stock={self.carbon_kgco2:.12g} kgCO2, '
                f'gap={carbon_gap:.12g} kgCO2, n_cohorts={len(self.q)}'
            )
        if self.carbon_kgco2 < -CARBON_BALANCE_ATOL_KGCO2:
            raise ValueError(f'Negative carbon stock in {self.name} at {stamp}')


def carbon_to_destinations(total_carbon, withdrawals):
    """Mass-proportional attribution for simultaneous withdrawals.

    All solver-fixed withdrawal masses are nonnegative after the common
    numerical input guard. A zero-mass destination MUST receive zero carbon.
    To avoid a negative remainder on an empty final destination (a subtle
    issue with the V5 implementation), assign the rounding remainder to the
    largest positive-mass destination. Carbon and proportionality are retained
    to machine precision, with no invented within-hour consumption ordering.
    """
    masses = {key: assert_nonnegative(float(value), f'withdrawal to {key}')
              for key, value in withdrawals.items()}
    if not np.isfinite(total_carbon) or total_carbon < -ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2:
        raise ValueError(f'Invalid carbon attributed to withdrawals: {total_carbon}')
    total_carbon = float(max(0.0, total_carbon))
    positive = [key for key, mass in masses.items() if mass > COHORT_TINY_MASS_KG]
    mass_total = sum(masses.values())
    if not positive:
        if total_carbon > ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2:
            raise ValueError('Carbon removed without material withdrawals')
        return {key: 0.0 for key in masses}
    largest = max(positive, key=lambda k: masses[k])
    out = {key: 0.0 for key in masses}
    for key in positive:
        if key != largest:
            out[key] = float(total_carbon * masses[key] / mass_total)
    out[largest] = float(total_carbon - sum(out.values()))
    if out[largest] < -ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2:
        raise ValueError('Negative proportional carbon allocation after rounding')
    if out[largest] < 0:
        # This branch is limited to sub-1e-8 kgCO2 machine arithmetic.
        out[largest] = 0.0
    return out


def run_fifo_lifo_ledger(recon_df, signal_name, signal_df, method):
    """Only the inventory withdrawal-carbon operation differs from the original code."""
    d = recon_df.merge(signal_df, on='datetime_utc', how='left', validate='one_to_one')
    if (~np.isfinite(pd.to_numeric(d['ef_kg_per_kwh'],errors='coerce'))).any():
        raise ValueError(f'Missing {signal_name} electricity emission factor after hourly merge')
    for unit in ('elz', 'nh3', 'meoh', 'crk'):
        d[f'elec_carbon_{unit}_kgph'] = 1000.0 * d[f'P_{unit}_grid_exact_mw'] * d['ef_kg_per_kwh']
    pools = {name: CohortPool(name, method, float(d.loc[0, f'tank_{name}_kg']))
             for name in POOL_NAMES}
    hourly_rows, audit_rows, pool_rows, cohort_trace, numerical_residual_events = [], [], [], [], []
    mass_correction_events = []
    mass_reconciliation_events = []
    for i, r in d.iterrows():
        ts = r['datetime_utc']
        opening_carbon = sum(pool.carbon_kgco2 for pool in pools.values())
        reconciliation_carbon_change = 0.0
        opening_mass_gap = {}
        opening_mass_gap_before_reconciliation = {}
        for name, pool in pools.items():
            solver_opening = float(r[f'tank_{name}_kg'])
            carbon_before_reconcile = pool.carbon_kgco2
            raw_gap = solver_opening - pool.mass_kg
            opening_mass_gap_before_reconciliation[name] = raw_gap
            correction = pool.reconcile_solver_opening(solver_opening, ts, sum(max(0.0,float(r[f'tank_{name.split(chr(95))[0]}_{x}_kg'])) for x in LABELS))
            reconciliation_carbon_change += pool.reconciliation_carbon_change
            if abs(correction) > MASS_RECON_LOG_MIN_KG or pool.dust_mass_removed > 0 or abs(pool.reconciliation_carbon_change) > 0:
                mass_reconciliation_events.append({
                    'datetime_utc': ts,
                    'Accounting method': method,
                    'Evaluation signal': signal_name,
                    'pool': name,
                    'solver_opening_mass_kg': solver_opening,
                    'cohort_mass_before_reconciliation_kg': solver_opening - raw_gap,
                    'unresolved_dust_gap_kg': solver_opening - pool.mass_kg,
                    'dust_mass_removed_kg': pool.dust_mass_removed,
                    'carbon_reconciliation_change_kgco2': pool.reconciliation_carbon_change,
                    'signed_mass_adjustment_kg': correction,
                    'absolute_mass_adjustment_kg': abs(correction),
                    'relative_mass_adjustment': abs(correction)/max(1.0, solver_opening),
                    'relative_to_actual_label_mass': abs(correction)/solver_opening if solver_opening>0 else np.nan,
                    'relative_to_physical_product_mass': abs(correction)/max(1.0,sum(max(0.0,float(r[f"tank_{name.split('_')[0]}_{x}_kg"])) for x in LABELS)),
                    'carbon_stock_before_kgco2': carbon_before_reconcile,
                    'carbon_stock_after_kgco2': pool.carbon_kgco2,
                    'method': RECONCILIATION_POLICY + ('; empty-state dust' if pool.dust_mass_removed else ''),
                })
            opening_mass_gap[name] = check_close_mass(
                pool.mass_kg, solver_opening, f'{method} {name} reconciled opening at {ts}')
        # Original archived rule: last-hour cracker split invalid without next state.
        cracker_valid = bool(r['Q_crk_split_valid'])
        if not cracker_valid and bool(r['has_next_state']) and r['crk_nh3_in_kgph'] > TOL:
            raise ValueError(f'Unexpected internal invalid cracker split at {ts}')
        flow_corrected = 0.0
        # V6: a single normalization boundary for ALL source material flows.
        # Previously, additions used clipped nonnegative masses while the
        # associated process-carbon terms still used signed raw values.
        # This created the M2 green-MeOH negative added-carbon failure.
        # Do not edit any source CSV values or solver material balances.
        mass_roundoff_this_hour = 0.0
        mass_roundoff_events_this_hour = 0
        def physical_mass(field, raw):
            nonlocal mass_roundoff_this_hour, mass_roundoff_events_this_hour
            raw = float(raw)
            clean = assert_nonnegative(raw, f'{field} at {ts}')
            if raw < 0.0:
                mass_roundoff_this_hour += -raw
                mass_roundoff_events_this_hour += 1
                mass_correction_events.append({
                    'datetime_utc': ts,
                    'Accounting method': method,
                    'Evaluation signal': signal_name,
                    'source_field': field,
                    'raw_mass_kgph': raw,
                    'used_nonnegative_mass_kgph': clean,
                    'numerical_mass_correction_kg': clean - raw,
                    'reason': 'Solver/reconstruction negative mass within 0.001 kg/h numerical limit',
                })
            return clean

        # The same production numbers must drive material inventory additions,
        # synthesis transfer stoichiometry, and direct-process carbon inputs.
        production = {
            (p, lab): physical_mass(f'{p}_prod_{lab}_kgph', r[f'{p}_prod_{lab}_kgph'])
            for p in PRODUCTS for lab in LABELS
        }
        sales = {
            (p, lab): physical_mass(f'{p}_sell_{lab}_kgph', r[f'{p}_sell_{lab}_kgph'])
            for p in PRODUCTS for lab in LABELS
        }
        flows = {}
        for label in LABELS:
            q_raw = float(r[f'Q_crk_in_{label}_recon_kgph']) if cracker_valid else 0.0
            if q_raw < 0:
                flow_corrected += -q_raw
            q = physical_mass(f'Q_crk_in_{label}_recon_kgph', q_raw)
            flows[label] = {
                'NH3_cracker_feed': q,
                'NH3_return': K_NH3_REC * q,
                'H2_cracker_out': K_H2_OUT * q,
                'H2_cracker_fuel': K_H2_FUEL * q,
                'H2_to_NH3': physical_mass(f'H_to_NH3_{label}_kgph', r[f'H_to_NH3_{label}_kgph']),
                'H2_to_MeOH': physical_mass(f'H_to_MeOH_{label}_kgph', r[f'H_to_MeOH_{label}_kgph']),
            }
        # First take ALL same-hour withdrawals using opening cohorts (no new output can sell now).
        outgoing_carbon = {}
        sold_burdens = {}
        fuel_sink = 0.0
        for label in LABELS:
            f = flows[label]
            h2_out = {
                'sold': sales[('H2', label)],
                'to_NH3': f['H2_to_NH3'],
                'to_MeOH': f['H2_to_MeOH'],
                'cracker_fuel': f['H2_cracker_fuel'],
            }
            nh3_out = {
                'sold': sales[('NH3', label)],
                'cracker_feed': f['NH3_cracker_feed'],
            }
            meoh_out = {'sold': sales[('MeOH', label)]}
            for p, destinations in (('H2', h2_out), ('NH3', nh3_out), ('MeOH', meoh_out)):
                total_kg = sum(destinations.values())
                carbon_total = pools[f'{p}_{label}'].withdraw(total_kg, ts, cohort_trace)
                attributed = carbon_to_destinations(carbon_total, destinations)
                for dest, carbon in attributed.items():
                    outgoing_carbon[(p, label, dest)] = carbon
                sold_burdens[(p, label)] = attributed['sold']
            fuel_sink += outgoing_carbon[('H2', label, 'cracker_fuel')]

        # EXACT original process-burden logic, fed by cohort-derived H2/NH3 withdrawals.
        additions = {}
        for label in LABELS:
            f = flows[label]
            q = f['NH3_cracker_feed']
            c_feed = outgoing_carbon[('NH3', label, 'cracker_feed')]
            # NH3 return inherits the proportional part of the feed burden; the remainder
            # goes to recovered H2, with cracker grid electricity added to NON-GREEN H2.
            c_return = c_feed * K_NH3_REC
            c_h2_cracked = c_feed - c_return
            c_h2_new = float(r[f'elec_carbon_elz_kgph']) if label == 'ng' else 0.0
            c_nh3_new = (
                outgoing_carbon[('H2', label, 'to_NH3')]
                + DE_NH3 * production[('NH3', label)]
            )
            c_meoh_new = (
                outgoing_carbon[('H2', label, 'to_MeOH')]
                + DE_MEOH * production[('MeOH', label)]
            )
            if label == 'ng':
                c_nh3_new += float(r['elec_carbon_nh3_kgph'])
                c_meoh_new += float(r['elec_carbon_meoh_kgph'])
                if cracker_valid:
                    c_h2_cracked += float(r['elec_carbon_crk_kgph'])
            additions[f'H2_{label}'] = (
                production[('H2', label)] + f['H2_cracker_out'],
                c_h2_new + c_h2_cracked,
            )
            additions[f'NH3_{label}'] = (
                production[('NH3', label)] + f['NH3_return'],
                c_nh3_new + c_return,
            )
            additions[f'MeOH_{label}'] = (production[('MeOH', label)], c_meoh_new)
        # Diagnose any remaining negative inherited burden before creating cohorts.
        # Clamping a negative carbon result here would conceal a potentially
        # incorrect allocation; V6 corrects source material residuals instead.
        for name, (mass, carbon) in additions.items():
            if carbon < 0.0:
                p, lab = name.split('_')
                inherited = (outgoing_carbon[('H2', lab, 'to_MeOH')]
                             if p == 'MeOH' else
                             outgoing_carbon[('H2', lab, 'to_NH3')]
                             if p == 'NH3' else float('nan'))
                direct = ((DE_MEOH if p == 'MeOH' else DE_NH3)
                          * production[(p, lab)]) if p != 'H2' else float('nan')
                raw_source = float(r[f'{p}_prod_{lab}_kgph'])
                raise ValueError(
                    f'Negative assigned carbon in {name} at {ts}: '
                    f'{carbon:.12g} kgCO2, added mass={mass:.12g} kg; '
                    f'original {p}_prod_{lab}={raw_source:.12g} kg/h, '
                    f'clean production={production[(p,lab)]:.12g} kg/h, '
                    f'inherited carbon={inherited:.12g}, direct carbon={direct:.12g}; '
                    'investigate inherited/energy contributions; no automatic carbon clipping')

        # V3: match any carbon-only addition against its corresponding, already
        # computed grid-electricity source. For H2, both electrolysis and (when
        # reconstructable) cracker electricity are assigned to recovered H2.
        # Green pools never carry a grid-carbon exception. Physical mass and
        # normal cohort carbon attribution remain entirely unchanged.
        allowable_grid_residuals = {
            'H2_ng': float(r['elec_carbon_elz_kgph'])
                     + (float(r['elec_carbon_crk_kgph']) if cracker_valid else 0.0),
            'NH3_ng': float(r['elec_carbon_nh3_kgph']),
            'MeOH_ng': float(r['elec_carbon_meoh_kgph']),
        }
        suppressed_this_hour = 0.0
        # Add each product/label's total hour-t output as ONE new cohort, available at t+1.
        for name, (mass, carbon) in additions.items():
            numerical_exclusion = pools[name].add(
                mass, carbon, ts,
                source_grid_carbon_residual=allowable_grid_residuals.get(name))
            if numerical_exclusion > 0:
                suppressed_this_hour += numerical_exclusion
                numerical_residual_events.append({
                    'datetime_utc': ts,
                    'Accounting method': method,
                    'Evaluation signal': signal_name,
                    'pool': name,
                    'new_material_mass_kg': float(mass),
                    'unassigned_source_carbon_kgco2': float(carbon),
                    'matched_grid_electricity_carbon_kgco2': allowable_grid_residuals.get(name, np.nan),
                    'excluded_numerical_carbon_kgco2': numerical_exclusion,
                    'reason': (
                        'Zero-mass addition: sub-1e-8 kgCO2 arithmetic roundoff'
                        if numerical_exclusion <= ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2
                        else 'Zero-mass addition: matching non-green source-grid electricity numerical residual'
                    ),
                })
        closing_carbon = sum(pool.carbon_kgco2 for pool in pools.values())
        total_carbon_sold = sum(sold_burdens.values())
        electricity_in = sum(float(r[f'elec_carbon_{u}_kgph']) for u in ('elz', 'nh3', 'meoh', 'crk'))
        direct_in = sum(DE_NH3 * production[('NH3', lab)]
                        + DE_MEOH * production[('MeOH', lab)] for lab in LABELS)
        # The original skipped last-hour cracker split and therefore its electricity
        # is NOT assigned to recovered H2 (report this unassigned amount explicitly).
        unassigned_cracker_electricity = (
            float(r['elec_carbon_crk_kgph']) if not cracker_valid else 0.0)
        residual = closing_carbon - opening_carbon - (
            electricity_in + direct_in - total_carbon_sold - fuel_sink
            - unassigned_cracker_electricity - suppressed_this_hour + reconciliation_carbon_change)
        scale = max(1.0, abs(opening_carbon), abs(closing_carbon), abs(electricity_in))
        if abs(residual) > CARBON_BALANCE_ATOL_KGCO2 + CARBON_BALANCE_RTOL * scale:
            raise ValueError(f'Carbon conservation failed {method} {ts}: residual={residual}')

        next_mass_gap = {}
        if i + 1 < len(d):
            for name, pool in pools.items():
                next_mass_gap[name] = check_close_mass(
                    pool.mass_kg, float(d.loc[i + 1, f'tank_{name}_kg']),
                    f'{method} {name} closing at {ts}')
        else:
            next_mass_gap = {name: np.nan for name in POOL_NAMES}
        # Record detailed cohort-state checks each hour; no claim about final closing stock.
        for name, pool in pools.items():
            pool.verify_internal_sums(ts)
            pool_rows.append({
                'datetime_utc': ts, 'method': method, 'pool': name,
                'solver_opening_mass_kg': float(r[f'tank_{name}_kg']),
                'cohort_opening_mass_gap_kg': opening_mass_gap[name],
                'cohort_opening_mass_gap_before_reconciliation_kg': opening_mass_gap_before_reconciliation[name],
                'cohort_after_hour_mass_kg': pool.mass_kg,
                'cohort_after_hour_carbon_kgco2': pool.carbon_kgco2,
                'cohort_count_after_hour': len(pool.q),
                'withdrawal_shortfall_cumulative_kg': pool.roundoff_kg,
                'withdrawal_shortfall_count_cumulative': pool.roundoff_count,
                'internal_mass_sum_gap_kg': sum(b.mass_kg for b in pool.q)-pool.mass_kg,
                'internal_carbon_sum_gap_kgco2': sum(b.carbon_kgco2 for b in pool.q)-pool.carbon_kgco2,
                'oldest_remaining_cohort_entry': str(pool.q[0].timestamp) if pool.q else '',
                'newest_remaining_cohort_entry': str(pool.q[-1].timestamp) if pool.q else '',
                'next_solver_opening_mass_gap_kg': next_mass_gap[name],
                'next_state_available': i + 1 < len(d),
            })
        sold_row = {}
        for p in PRODUCTS:
            gmass = sales[(p, 'g')]
            ngmass = sales[(p, 'ng')]
            total_mass = gmass + ngmass
            gcarbon = sold_burdens[(p, 'g')]
            ngcarbon = sold_burdens[(p, 'ng')]
            sold_row.update({
                f'{p}_sold_g_kgph': gmass,
                f'{p}_sold_ng_kgph': ngmass,
                f'{p}_sold_total_kgph': total_mass,
                f'C_sell_{p}_g_kgco2': gcarbon,
                f'C_sell_{p}_ng_kgco2': ngcarbon,
                f'{p}_sold_CI_kgco2_perkg_hourly': (
                    (gcarbon + ngcarbon) / total_mass if total_mass > 0 else np.nan),
            })
        # Internal inherited burdens are exported for scientific inspection;
        # they are NOT used to alter the optimizer's fixed flows.
        internal = {}
        for lab in LABELS:
            for p, dests in {
                'H2': ('to_NH3','to_MeOH','cracker_fuel'),
                'NH3': ('cracker_feed',),
            }.items():
                for dest in dests:
                    internal[f'C_{p}_{lab}_{dest}_kgco2'] = outgoing_carbon[(p,lab,dest)]
            internal[f'C_added_H2_{lab}_kgco2'] = additions[f'H2_{lab}'][1]
            internal[f'C_added_NH3_{lab}_kgco2'] = additions[f'NH3_{lab}'][1]
            internal[f'C_added_MeOH_{lab}_kgco2'] = additions[f'MeOH_{lab}'][1]
        hourly_rows.append({
            'datetime_utc': ts, 'day_of_year': int(r['day_of_year']),
            'hour_in_day': int(r['hour_in_day']), 'Evaluation signal': signal_name,
            'Accounting method': method, **sold_row, **internal,
            'Q_crk_split_valid': cracker_valid,
            'last_hour_cracker_reconstruction_unavailable': not bool(r['has_next_state']),
            'unassigned_last_hour_cracker_electricity_kgco2': unassigned_cracker_electricity,
            'excluded_numerical_carbon_residual_kgco2': suppressed_this_hour,
        })
        audit_rows.append({
            'datetime_utc': ts, 'Accounting method': method,
            'pool_opening_max_abs_mass_gap_kg': max(abs(x) for x in opening_mass_gap.values()),
            'pool_opening_max_abs_gap_before_reconciliation_kg': max(abs(x) for x in opening_mass_gap_before_reconciliation.values()),
            'pool_closing_max_abs_mass_gap_kg': (max(abs(x) for x in next_mass_gap.values())
                                                 if i + 1 < len(d) else np.nan),
            'opening_carbon_stock_kgco2': opening_carbon,
            'closing_carbon_stock_accounted_kgco2': closing_carbon if i + 1 < len(d) else np.nan,
            'new_electricity_carbon_kgco2': electricity_in,
            'new_direct_emission_kgco2': direct_in,
            'sold_product_carbon_kgco2': total_carbon_sold,
            'cracker_fuel_carbon_sink_kgco2': fuel_sink,
            'unassigned_cracker_electricity_kgco2': unassigned_cracker_electricity,
            'excluded_numerical_carbon_residual_kgco2': suppressed_this_hour,
            'excluded_numerical_carbon_residual_kgco2_cumulative': (
                sum(p.suppressed_numerical_carbon_kgco2 for p in pools.values())),
            'plant_carbon_balance_residual_kgco2': residual,
            'carbon_reconciliation_change_kgco2': reconciliation_carbon_change,
            'cracker_feed_split_valid': cracker_valid,
            'corrected_small_negative_cracker_feed_kg': flow_corrected,
            'corrected_small_negative_all_source_flows_kg': mass_roundoff_this_hour,
            'n_corrected_small_negative_all_source_flows': mass_roundoff_events_this_hour,
            'cohort_withdrawal_roundoff_kg_cumulative': sum(p.roundoff_kg for p in pools.values()),
        })
    for pool in pools.values():
        pool.verify_internal_sums('end of account')
    hourly = pd.DataFrame(hourly_rows)
    audit = pd.DataFrame(audit_rows)
    pool_audit = pd.DataFrame(pool_rows)
    final_counts = pd.DataFrame([{
        'Accounting method': method, 'Evaluation signal': signal_name,
        'Pool': name, 'final_cohort_count': len(p.q),
        'last_hour_stock_unverified': True,
        'withdrawal_roundoff_kg': p.roundoff_kg,
        'withdrawal_roundoff_events': p.roundoff_count,
        'excluded_numerical_carbon_residual_kgco2_total': p.suppressed_numerical_carbon_kgco2,
        'excluded_numerical_carbon_residual_events': p.suppressed_numerical_carbon_events,
    } for name, p in pools.items()])
    trace_frame = pd.DataFrame(cohort_trace)
    residual_frame = pd.DataFrame(numerical_residual_events, columns=[
        'datetime_utc', 'Accounting method', 'Evaluation signal', 'pool',
        'new_material_mass_kg', 'unassigned_source_carbon_kgco2',
        'matched_grid_electricity_carbon_kgco2',
        'excluded_numerical_carbon_kgco2', 'reason'])
    mass_correction_frame = pd.DataFrame(mass_correction_events, columns=[
        'datetime_utc', 'Accounting method', 'Evaluation signal', 'source_field',
        'raw_mass_kgph', 'used_nonnegative_mass_kgph',
        'numerical_mass_correction_kg', 'reason'])
    reconciliation_frame = pd.DataFrame(mass_reconciliation_events, columns=[
        'datetime_utc', 'Accounting method', 'Evaluation signal', 'pool',
        'solver_opening_mass_kg', 'cohort_mass_before_reconciliation_kg',
        'signed_mass_adjustment_kg', 'absolute_mass_adjustment_kg',
        'relative_mass_adjustment', 'relative_to_actual_label_mass', 'relative_to_physical_product_mass', 'unresolved_dust_gap_kg', 'dust_mass_removed_kg',
        'carbon_reconciliation_change_kgco2', 'carbon_stock_before_kgco2',
        'carbon_stock_after_kgco2', 'method'])
    return (hourly, audit, pool_audit, final_counts, trace_frame,
            residual_frame, mass_correction_frame, reconciliation_frame)


# ============================================================
# REPORTING - sales-weighted annual CI; original baseline NOT recomputed
# ============================================================
def summarize_daily(hourly):
    rows = []
    for day, group in hourly.groupby('day_of_year', sort=True):
        row = {
            'day_of_year': int(day), 'date': str(group['datetime_utc'].iloc[0].date()),
            'Accounting method': group['Accounting method'].iloc[0],
            'Evaluation signal': group['Evaluation signal'].iloc[0],
        }
        for p in PRODUCTS:
            mass = group[f'{p}_sold_total_kgph'].sum()
            carbon = group[f'C_sell_{p}_g_kgco2'].sum() + group[f'C_sell_{p}_ng_kgco2'].sum()
            row[f'{p}_sold_total_kg'] = mass
            row[f'{p}_sold_burden_kgco2'] = carbon
            row[f'{p}_sold_CI_kgco2_perkg'] = carbon / mass if mass > TOL else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_annual(hourly, model_name):
    row = {
        'Model': model_name, 'Accounting method': hourly['Accounting method'].iloc[0],
        'Evaluation signal': hourly['Evaluation signal'].iloc[0],
        'period_start_utc': str(hourly['datetime_utc'].iloc[0]),
        'period_end_utc': str(hourly['datetime_utc'].iloc[-1]),
        'last_hour_closing_inventory_verified': False,
    }
    for p in PRODUCTS:
        mass = float(hourly[f'{p}_sold_total_kgph'].sum())
        carbon = float(hourly[f'C_sell_{p}_g_kgco2'].sum()
                       + hourly[f'C_sell_{p}_ng_kgco2'].sum())
        row[f'{p}_sold_mass_kg'] = mass
        row[f'{p}_sold_burden_kgco2'] = carbon
        row[f'{p}_sold_CI_kgco2_perkg'] = carbon / mass if mass > TOL else np.nan
        row[f'{p}_sold_green_share_pct'] = (
            float(hourly[f'{p}_sold_g_kgph'].sum()) / mass * 100 if mass > TOL else np.nan)
    return row


def paired_annual(annual):
    rows = []
    for (method, signal), group in annual.groupby(['Accounting method', 'Evaluation signal']):
        baseline = group.loc[group['Model'].eq('M0')]
        if baseline.empty:
            continue
        base = baseline.iloc[0]
        for _, row in group.loc[~group['Model'].eq('M0')].iterrows():
            record = {'Accounting method': method, 'Evaluation signal': signal,
                      'Model': row['Model'], 'Baseline': 'M0'}
            for p in PRODUCTS:
                m0 = float(base[f'{p}_sold_CI_kgco2_perkg'])
                m2 = float(row[f'{p}_sold_CI_kgco2_perkg'])
                record[f'{p}_M0_CI_kgco2_perkg'] = m0
                record[f'{p}_M2_CI_kgco2_perkg'] = m2
                record[f'{p}_CI_reduction_vs_M0_pct'] = 100 * (1-m2/m0) if m0 > TOL else np.nan
            rows.append(record)
    return pd.DataFrame(rows)


def hourly_distribution(all_hourly):
    rows = []
    for (model, signal, method), g in all_hourly.groupby(['Model', 'Evaluation signal', 'Accounting method']):
        for p in PRODUCTS:
            s = g[f'{p}_sold_CI_kgco2_perkg_hourly'].dropna()
            rows.append({'Model': model, 'Evaluation signal': signal,
                         'Accounting method': method, 'Product': p,
                         'n_positive_sales_hours': int(len(s)),
                         'mean_hourly_CI': float(s.mean()) if len(s) else np.nan,
                         'std_hourly_CI': float(s.std(ddof=0)) if len(s) else np.nan,
                         'P05_hourly_CI': float(s.quantile(.05)) if len(s) else np.nan,
                         'P50_hourly_CI': float(s.quantile(.50)) if len(s) else np.nan,
                         'P95_hourly_CI': float(s.quantile(.95)) if len(s) else np.nan})
    return pd.DataFrame(rows)


def paired_hourly(all_hourly):
    rows = []
    groups = all_hourly.groupby(['Accounting method', 'Evaluation signal'])
    for (method, signal), group in groups:
        baseline = group[group.Model.eq('M0')]
        if baseline.empty:
            continue
        for model in group.loc[~group.Model.eq('M0'), 'Model'].unique():
            other = group[group.Model.eq(model)]
            merged = baseline.merge(other, on='datetime_utc', suffixes=('_m0','_m2'))
            for p in PRODUCTS:
                c = f'{p}_sold_CI_kgco2_perkg_hourly'
                valid = merged[f'{c}_m0'].notna() & merged[f'{c}_m2'].notna()
                delta = (merged.loc[valid, f'{c}_m2'] - merged.loc[valid, f'{c}_m0'])
                rows.append({'Accounting method': method, 'Evaluation signal': signal,
                             'Model': model, 'Product': p,
                             'n_paired_positive_sales_hours': int(valid.sum()),
                             'share_hours_M2_higher_than_M0_pct': (
                                 100 * float((delta > 1e-12).mean()) if len(delta) else np.nan),
                             'deltaCI_mean_kgco2_perkg': float(delta.mean()) if len(delta) else np.nan,
                             'deltaCI_std_kgco2_perkg': float(delta.std(ddof=0)) if len(delta) else np.nan,
                             'deltaCI_P05_kgco2_perkg': float(delta.quantile(.05)) if len(delta) else np.nan,
                             'deltaCI_P50_kgco2_perkg': float(delta.quantile(.50)) if len(delta) else np.nan,
                             'deltaCI_P95_kgco2_perkg': float(delta.quantile(.95)) if len(delta) else np.nan})
    return pd.DataFrame(rows)


# ============================================================
# AUTOMATED TESTS — entirely synthetic, independent of manuscript numbers
# ============================================================
def self_tests():
    records = []
    def test(name, fn):
        try:
            fn()
            records.append({'test': name, 'status': 'PASS', 'details': 'Controlled synthetic test'})
        except Exception as ex:
            records.append({'test': name, 'status': 'FAIL', 'details': str(ex)})
            raise

    def t_fifo_lifo():
        for method, expected in [('FIFO', 50.0), ('LIFO', 210.0)]:
            p = CohortPool('synthetic', method, 0)
            p.add(60, 60, 'old'); p.add(40, 200, 'new')
            carbon = p.withdraw(50, 'later')
            assert abs(carbon-expected) < 1e-9
            assert abs(p.mass_kg-50) < 1e-9
            assert abs(p.carbon_kgco2-(260-expected)) < 1e-9
            p.verify_internal_sums('later')
    test('FIFO/LIFO distinct chronological withdrawals and carbon preservation', t_fifo_lifo)

    def t_proportions():
        allocations = carbon_to_destinations(3000, {'sold':500,'NH3':300,'MeOH':200})
        assert abs(allocations['sold'] - 1500) < 1e-9
        assert abs(allocations['NH3'] - 900) < 1e-9
        assert abs(allocations['MeOH'] - 600) < 1e-9
        assert abs(sum(allocations.values())-3000)<1e-9
    test('Simultaneous process/sale withdrawals allocated proportionally', t_proportions)

    def t_midnight():
        p = CohortPool('synthetic', 'FIFO', 5)
        p.add(10, 20, '2024-01-01 23:00')
        p.add(10, 60, '2024-01-02 00:00')
        p.withdraw(5, '2024-01-02 20:00')
        carbon = p.withdraw(10, '2024-01-03 10:00')
        assert abs(carbon-20)<1e-9
        assert abs(p.carbon_kgco2-60)<1e-9
    test('No reset at midnight; FIFO carries cohorts across days', t_midnight)

    def t_same_hour():
        p = CohortPool('synthetic', 'LIFO', 10)
        assert abs(p.withdraw(5,'hour_1')-0)<1e-12
        p.add(8,24,'hour_1_end')
        assert abs(p.withdraw(8,'hour_2')-24)<1e-12
    test('Incoming material unavailable until next hour', t_same_hour)

    def t_recovery():
        c_feed, q = 200.0, 100.0
        c_return = c_feed*K_NH3_REC
        c_h2 = c_feed - c_return + 10.0
        assert abs(c_return+c_h2-(c_feed+10)) < 1e-9
        assert abs(K_H2_OUT*q-12.024) < 1e-9
    test('Ammonia-cracker carbon inheritance and electricity conservation', t_recovery)

    def t_numerical_carbon_only_residue():
        # Reproduces the magnitude reported by the author on 2024-03-08:
        # H2 production = 0, recovered H2 = 0, source-grid electricity carbon
        # = 9.628539488718421e-5 kgCO2. This is an accounting-numerics test,
        # not a recalculation of any research or SI result.
        residual = 9.628539488718421e-5
        for mode in METHODS:
            pool = CohortPool('H2_ng', mode, 10.0)
            before_mass, before_carbon = pool.mass_kg, pool.carbon_kgco2
            excluded = pool.add(0.0, residual, '2024-03-08T00:00Z',
                                source_grid_carbon_residual=residual)
            assert excluded == residual
            assert pool.mass_kg == before_mass and pool.carbon_kgco2 == before_carbon
            assert pool.suppressed_numerical_carbon_kgco2 == residual
            # The excluded source carbon is an explicit sink in the audit:
            # closing-opening - (electricity - exclusion) == 0.
            assert abs((pool.carbon_kgco2-before_carbon)-(residual-excluded)) < 1e-12
            pool.verify_internal_sums('after numerical exclusion')
            for wrong_carbon, source in [(1.01e-3,1.01e-3),
                                         (residual, 0.0),
                                         (residual, None)]:
                try:
                    pool.add(0.0, wrong_carbon, 'bad',
                             source_grid_carbon_residual=source)
                except ValueError:
                    pass
                else:
                    raise AssertionError('Invalid carbon-only addition was accepted')
    test('Traceable 0.001 kgCO2 carbon-only grid residual; reject unrelated carbon',
         t_numerical_carbon_only_residue)

    def t_green_arithmetic_roundoff():
        # Reproduce only the magnitude shown by the author; do not use any
        # optimization observations or recalculate a manuscript result.
        tiny = 1.4012992352085973e-9
        for mode in METHODS:
            pool = CohortPool('H2_g', mode, 20.0)
            excluded = pool.add(0.0, tiny, '2024-02-06 01:00 UTC')
            assert excluded == tiny
            assert pool.mass_kg == 20.0
            assert pool.carbon_kgco2 == 0.0
            assert pool.suppressed_numerical_carbon_events == 1
            assert abs(pool.suppressed_numerical_carbon_kgco2-tiny) < 1e-20
            pool.verify_internal_sums('after green-roundoff suppression')
            # Do not accept a larger green carbon-only burden without mass.
            try:
                pool.add(0.0, 1.01e-8, 'larger-unattributed-green-burden')
            except ValueError:
                pass
            else:
                raise AssertionError('Accepted non-negligible green carbon without mass')
        # At the threshold, a tiny nonzero mass is treated consistently.
        p = CohortPool('H2_g', 'FIFO', 1)
        assert p.add(COHORT_TINY_MASS_KG, 1e-9, 'mass-zero-tolerance') == 1e-9
    test('Green-pool ~1.4e-9 kgCO2 roundoff: audited suppression, stricter error above 1e-8',
         t_green_arithmetic_roundoff)

    def t_no_independent_carbon_reset():
        # V4 bug regression: small but NONZERO carbon remains represented in
        # a partial cohort. The running total must not independently snap
        # to zero, especially after many additions and partial withdrawals.
        for mode in METHODS:
            p = CohortPool('MeOH_ng', mode, 0.0)
            for i in range(100):
                p.add(10.0, 9e-6, f'hour{i}_addition')
                p.withdraw(1.0, f'hour{i}_withdrawal')
                p.verify_internal_sums(f'hour{i}_audit')
            cohort_carbon = sum(b.carbon_kgco2 for b in p.q)
            assert cohort_carbon > 0
            assert abs(cohort_carbon - p.carbon_kgco2) < 1e-12
    test('V5: preserve small real cohort-carbon stocks over repeated partial withdrawals',
         t_no_independent_carbon_reset)

    def t_residual_in_complete_ledger():
        # Full hourly end-to-end case, with solver-like nonzero grid power
        # but zero electrolyzer product mass. Tests both the audited source
        # exclusion and the strict hourly conservation formula, not only add().
        opening = {name: 10.0 for name in POOL_NAMES}
        frame_rows = []
        for hour in range(3):
            row = {col: 0.0 for col in REQUIRED_MODEL_COLS}
            row.update({
                'model_tag': 'synthetic_numerical_residual',
                'datetime_utc': pd.Timestamp('2024-03-08', tz='UTC') + pd.Timedelta(hours=hour),
                'day_of_year': 68, 'hour_in_day': hour,
                'P_elz_mw': 1.0263431349244456e-6 if hour == 0 else 0.0,
            })
            for name, mass in opening.items():
                row[f'tank_{name}_kg'] = mass
            frame_rows.append(row)
        frame, _ = reconstruct_unit_grid_battery_split(pd.DataFrame(frame_rows))
        frame, _ = reconstruct_intermediates(frame)
        sig = pd.DataFrame({
            'datetime_utc': frame['datetime_utc'],
            'ef_kg_per_kwh': [9.628539488718421e-5 / (1000 * 1.0263431349244456e-6)] * 3
        })
        for mode in METHODS:
            hourly, audit, pool, final, trace, residuals, mass_residuals, mass_reconciliations = run_fifo_lifo_ledger(
                frame, 'synthetic_grid_residual', sig, mode)
            assert len(residuals) == 1
            assert residuals.iloc[0]['pool'] == 'H2_ng'
            assert abs(float(residuals.iloc[0]['excluded_numerical_carbon_kgco2'])
                       - 9.628539488718421e-5) < 1e-12
            assert abs(audit['plant_carbon_balance_residual_kgco2']).max() < 1e-12
            assert abs(audit['pool_closing_max_abs_mass_gap_kg'].dropna()).max() < 1e-12
            assert abs(audit['excluded_numerical_carbon_residual_kgco2'].sum()
                       - 9.628539488718421e-5) < 1e-12
    test('End-to-end zero-product grid residual: event log and exact conservation',
         t_residual_in_complete_ledger)

    def t_signed_source_flows_all_six_pools():
        # M2-like corner case: optimizer exports an insignificant negative
        # labeled production flow; the ORIGINAL V5 consumed it as a zero-mass
        # addition but still used its negative direct-emission burden.
        # This is NOT a test of experimental/manuscript CI results.
        for affected_p in PRODUCTS:
            for affected_lab in LABELS:
                raw_rows = []
                initial = {name: 100.0 for name in POOL_NAMES}
                for hour in range(3):
                    row = {col: 0.0 for col in REQUIRED_MODEL_COLS}
                    row.update({
                        'model_tag': 'synthetic_signed_flow',
                        'datetime_utc': pd.Timestamp('2024-04-21', tz='UTC') + pd.Timedelta(hours=hour),
                        'day_of_year': 112, 'hour_in_day': hour,
                    })
                    for name in POOL_NAMES:
                        row[f'tank_{name}_kg'] = initial[name]
                    if hour == 0:
                        row[f'{affected_p}_prod_{affected_lab}_kgph'] = -0.000223325
                        # Consistent solver-like next-hour opening tank states:
                        initial[f'{affected_p}_{affected_lab}'] -= 0.000223325
                        if affected_p == 'NH3':
                            initial[f'H2_{affected_lab}'] += M_H2_TO_NH3 * 0.000223325
                        elif affected_p == 'MeOH':
                            initial[f'H2_{affected_lab}'] += M_H2_TO_MEOH * 0.000223325
                    raw_rows.append(row)
                source = pd.DataFrame(raw_rows)
                source, _ = reconstruct_unit_grid_battery_split(source)
                source, _ = reconstruct_intermediates(source)
                ef = pd.DataFrame({
                    'datetime_utc': source['datetime_utc'],
                    'ef_kg_per_kwh': [0.2] * 3,
                })
                for mode in METHODS:
                    hourly, audit, pools, final, tr, residuals, mass_corrections, mass_reconciliations = run_fifo_lifo_ledger(
                        source, 'synthetic', ef, mode)
                    assert (mass_corrections['source_field'] == f'{affected_p}_prod_{affected_lab}_kgph').any()
                    assert abs(audit.plant_carbon_balance_residual_kgco2).max() < 1e-7
                    assert abs(audit.pool_closing_max_abs_mass_gap_kg.dropna()).max() < 0.001
                    assert (hourly[f'C_added_{affected_p}_{affected_lab}_kgco2'] >= -1e-12).all()
    test('V6: signed near-zero production, all six pools, FIFO and LIFO',
         t_signed_source_flows_all_six_pools)

    def t_proportional_last_empty_destination():
        a=carbon_to_destinations(100.0, {
            'sold': 50.0, 'to_NH3': 30.0, 'to_MeOH': 20.0, 'cracker_fuel': 0.0})
        assert a['cracker_fuel'] == 0.0
        assert abs(sum(a.values())-100.0) < 1e-12
        z=carbon_to_destinations(1.0e-5, {'sold': 10, 'cracker_feed': 0})
        assert z['cracker_feed'] == 0.0 and z['sold'] == 1e-5
    test('V6: proportional carbon allocation cannot attribute to zero-mass final flow',
         t_proportional_last_empty_destination)

    def t_full_engine():
        # Controlled 3-hour trajectory with both labels, synthesis and cracking;
        # opening inventory states are generated from the exact optimization balances.
        base = {col:0.0 for col in REQUIRED_MODEL_COLS}
        open_stock = {'H2_g':10.,'H2_ng':10.,'NH3_g':20.,'NH3_ng':20.,
                      'MeOH_g':5.,'MeOH_ng':5.}
        rs=[]
        for i in range(3):
            r=base.copy()
            r.update({'model_tag':'test', 'datetime_utc':pd.Timestamp('2024-01-01',tz='UTC')+pd.Timedelta(hours=i),
                      'day_of_year':1,'hour_in_day':i})
            for name in POOL_NAMES: r[f'tank_{name}_kg']=open_stock[name]
            if i==0:
                r.update({'H2_prod_g_kgph':3.,'H2_prod_ng_kgph':2.,
                          'NH3_prod_g_kgph':1.,'NH3_sell_ng_kgph':1.,
                          'MeOH_prod_g_kgph':4.,'H2_sell_g_kgph':1.,'H2_sell_ng_kgph':2.,
                          'crk_nh3_in_kgph':2.,
                          'crk_h2_out_kgph':K_H2_OUT*2,
                          'crk_h2_fuel_kgph':K_H2_FUEL*2})
            if i==1:
                r.update({'H2_sell_g_kgph':2.,'NH3_sell_g_kgph':1.,
                          'MeOH_sell_g_kgph':2.})
            rs.append(r)
            for lab in LABELS:
                q = 2. if i==0 and lab=='ng' else 0.
                name=f'H2_{lab}'
                open_stock[name] += (float(r[f'H2_prod_{lab}_kgph'])+K_H2_OUT*q
                                     -M_H2_TO_NH3*float(r[f'NH3_prod_{lab}_kgph'])
                                     -M_H2_TO_MEOH*float(r[f'MeOH_prod_{lab}_kgph'])
                                     -float(r[f'H2_sell_{lab}_kgph'])-K_H2_FUEL*q)
                name=f'NH3_{lab}'
                open_stock[name] += (float(r[f'NH3_prod_{lab}_kgph'])
                                     +K_NH3_REC*q-float(r[f'NH3_sell_{lab}_kgph'])-q)
                name=f'MeOH_{lab}'
                open_stock[name] += (float(r[f'MeOH_prod_{lab}_kgph'])
                                     -float(r[f'MeOH_sell_{lab}_kgph']))
        frame=pd.DataFrame(rs)
        frame,_=reconstruct_unit_grid_battery_split(frame)
        frame,_=reconstruct_intermediates(frame)
        assert frame.Q_crk_split_valid.iloc[:2].all()
        assert not frame.Q_crk_split_valid.iloc[2]
        sig=pd.DataFrame({'datetime_utc':frame.datetime_utc,
                          'ef_kg_per_kwh':[0.2,0.3,0.4]})
        for mode in METHODS:
            hourly,audit,pool,_,trace,residuals,mass_residuals,mass_reconciliations=run_fifo_lifo_ledger(frame,'synthetic',sig,mode)
            assert len(hourly)==3 and len(pool)==18 and len(trace)>0
            assert abs(audit.plant_carbon_balance_residual_kgco2).max()<1e-7
            assert abs(audit.pool_closing_max_abs_mass_gap_kg.dropna()).max()<1e-7
            assert np.isfinite(hourly['H2_sold_CI_kgco2_perkg_hourly'].iloc[0])
    def t_v7_reconcile_proportional():
        for mode in METHODS:
            pool=CohortPool('MeOH_ng',mode,200.0)
            pool.add(300.0,900.0,'2024-01-01 00:00')
            pool.add(336.1596938343719,1008.479081503116,'2024-01-01 01:00')
            original=[(z.mass_kg,z.carbon_kgco2,z.timestamp) for z in pool.q]
            total_before=pool.carbon_kgco2
            diff=pool.reconcile_solver_opening(836.1629836,'2024-06-24 09:00')
            assert abs(diff-0.0032897656281)<1e-9
            assert abs(pool.mass_kg-836.1629836)<1e-10
            assert pool.carbon_kgco2==total_before
            for item,(m,c,t) in zip(pool.q,original):
                assert item.carbon_kgco2==c and item.timestamp==t
                assert abs(item.mass_kg / m - 836.1629836/836.1596938343719)<1e-11
            pool.verify_internal_sums('after V7 reconciliation')
            assert pool.withdraw(836.1627913,'2024-06-24 09:00')>=0
    test('V7: 3.29g M2 opening drift repaired without carbon or time-order changes', t_v7_reconcile_proportional)

    def t_v7_reconcile_limits():
        pool=CohortPool('MeOH_ng','FIFO',836.1596938343719)
        try:
            pool.reconcile_solver_opening(836.200,'large gap')
        except ValueError as e:
            assert 'UNRESOLVED' in str(e)
        else:
            raise AssertionError('Large input discrepancy was not rejected')
        zero=CohortPool('H2_g','FIFO',0.0)
        try:
            zero.reconcile_solver_opening(0.0001,'empty pool')
        except ValueError as e:
            assert 'empty' in str(e)
        else:
            raise AssertionError('Untraceable inventory at empty pool was invented')
    test('V7: large or empty-pool discrepancies trigger explicit failure', t_v7_reconcile_limits)

    def t_v7_coupled_meoh_near_empty():
        # End-to-end test of the M2-like 3.29 g exported mass discrepancy.
        # Preserve the NEXT state's authoritative value and a near-exhausting
        # sale; do NOT use user research data or reproduce manuscript values.
        opening = {n:100.0 for n in POOL_NAMES}
        opening['MeOH_ng'] = 836.1596938343719
        rows=[]
        for hour in range(3):
            r={col:0.0 for col in REQUIRED_MODEL_COLS}
            r.update({'model_tag':'synthetic_m2_balance_gap',
                      'datetime_utc':pd.Timestamp('2024-06-24 08:00',tz='UTC')+pd.Timedelta(hours=hour),
                      'day_of_year':176, 'hour_in_day':8+hour})
            for name in POOL_NAMES:
                r[f'tank_{name}_kg']=opening[name]
            if hour==0:
                r['MeOH_prod_ng_kgph']=1.0
                opening['MeOH_ng'] += 1.0 + 0.0032897656281
                opening['H2_ng'] -= M_H2_TO_MEOH
            if hour==1:
                r['MeOH_sell_ng_kgph']=837.1627913
                opening['MeOH_ng'] -= 837.1627913
            rows.append(r)
        frame=pd.DataFrame(rows)
        frame,_=reconstruct_unit_grid_battery_split(frame)
        frame,_=reconstruct_intermediates(frame)
        source=material_source_balance_audit(frame)
        assert abs(source['raw_source_balance_gap_kg'].max()-0.0032897656281)<1e-8
        ef=pd.DataFrame({'datetime_utc':frame.datetime_utc,
                         'ef_kg_per_kwh':[0.2]*len(frame)})
        for mode in METHODS:
            hourly,audit,pool,_,trace,residuals,mass_residuals,recs=run_fifo_lifo_ledger(
                frame,'synthetic',ef,mode)
            assert (recs.pool=='MeOH_ng').any()
            assert abs(recs.loc[recs.pool=='MeOH_ng','signed_mass_adjustment_kg'].sum()-0.0032897656281)<1e-8
            assert abs(audit.plant_carbon_balance_residual_kgco2).max()<1e-8
            assert abs(audit.pool_closing_max_abs_mass_gap_kg.dropna()).max()<0.01
            assert np.isfinite(hourly['MeOH_sold_CI_kgco2_perkg_hourly'].iloc[1])
    test('V7: source-balance preflight and near-exhausting MeOH sale pass', t_v7_coupled_meoh_near_empty)

    test('Full coupled ledger: mass, cracker flows, temporal and carbon balance', t_full_engine)
    return pd.DataFrame(records)


# ============================================================
# MAIN EXECUTION / ZIP
# ============================================================
def source_stage_audit(raw, d):
    """Vectorized independent residual decomposition; no carbon calculation.

    Positive gap means saved next opening exceeds opening + inflow - outflow.
    Raw NH3 closure is an identity because labeled feed is inferred from it.
    'original_clipped' is before total-feed rescaling; 'rescaled' is before
    the common nonnegative-flow boundary. Both labels are retained separately.
    """
    frames=[]
    for lab in LABELS:
        qraw=(raw[f'NH3_prod_{lab}_kgph']-raw[f'NH3_sell_{lab}_kgph']
              -raw[f'tank_NH3_{lab}_kg'].diff().shift(-1))/(1-K_NH3_REC)
        qclip=clip_small_negative(qraw)
        qscale=d[f'Q_crk_in_{lab}_recon_kgph']
        for p in PRODUCTS:
            old=raw[f'tank_{p}_{lab}_kg']; nxt=old.shift(-1)
            prod=raw[f'{p}_prod_{lab}_kgph'];sale=raw[f'{p}_sell_{lab}_kgph']
            f=pd.DataFrame({'datetime_utc':raw.datetime_utc,'pool':f'{p}_{lab}',
                'opening_raw_kg':old,'next_opening_raw_kg':nxt,
                'production_raw_kg':prod,'sale_raw_kg':sale,
                'cracker_feed_raw_kg':qraw,'cracker_feed_original_clipped_kg':qclip,
                'cracker_feed_rescaled_kg':qscale,'cracker_feed_used_kg':qscale.clip(lower=0),
                'cracker_total_exported_kg':raw.crk_nh3_in_kgph,
                'cracker_total_gap_before_scaling_kg':d.Q_crk_total_gap_kgph,
                'NH3_return_used_kg':K_NH3_REC*qscale.clip(lower=0),
                'day_boundary':raw.hour_in_day.eq(23)})
            for stage,q,clean in [('raw',qraw,False),('original_clipped',qclip,False),
                                  ('rescaled',qscale,False),('sanitized',qscale.clip(lower=0),True)]:
                pp=prod.clip(lower=0) if clean else prod
                ss=sale.clip(lower=0) if clean else sale
                nh=raw[f'NH3_prod_{lab}_kgph'];me=raw[f'MeOH_prod_{lab}_kgph']
                if clean: nh=nh.clip(lower=0);me=me.clip(lower=0)
                extra=(-(1-K_NH3_REC)*q if p=='NH3' else
                       (K_H2_OUT-K_H2_FUEL)*q-M_H2_TO_NH3*nh-M_H2_TO_MEOH*me if p=='H2' else 0)
                f[f'{stage}_balance_gap_kg']=nxt-(old+pp-ss+extra)
            f['rescaling_effect_kg']=f.rescaled_balance_gap_kg-f.original_clipped_balance_gap_kg
            f['sanitization_effect_kg']=f.sanitized_balance_gap_kg-f.rescaled_balance_gap_kg
            f['negative_stock_normalization_kg']=(-old).clip(lower=0)
            f['positive_negative_flow_correction_kg']=(-prod).clip(lower=0)+(-sale).clip(lower=0)
            f['relative_sanitized_gap']=f.sanitized_balance_gap_kg.abs()/np.maximum(1,nxt.abs())
            frames.append(f.iloc[:-1])
    return pd.concat(frames,ignore_index=True)


def stage_summary(stages):
    rows=[]
    for pool,g in stages.groupby('pool'):
        for stage in ('raw','original_clipped','rescaled','sanitized'):
            s=g[f'{stage}_balance_gap_kg']; a=s.abs(); idx=a.idxmax()
            rows.append({'pool':pool,'stage':stage,'count':len(s),
                'n_abs_above_1e8_kg':int((a>1e-8).sum()),'min_signed_kg':s.min(),
                'max_signed_kg':s.max(),'max_abs_kg':a.max(),'p50_abs_kg':a.quantile(.5),
                'p95_abs_kg':a.quantile(.95),'p99_abs_kg':a.quantile(.99),
                'net_gap_kg':s.sum(),'sum_absolute_gaps_kg':a.sum(),
                'max_hour_utc':str(g.loc[idx,'datetime_utc']),
                'day_boundary_max_abs_kg':a[g.day_boundary].max()})
    return pd.DataFrame(rows)


def extra_tests():
    records=[]
    for mode in METHODS:
        p=CohortPool('equal_ci',mode,0)
        p.add(2,6,'old');p.add(3,9,'new')
        assert abs(p.withdraw(4,'now')-12)<1e-12
        p.verify_internal_sums('partial')
        p=CohortPool('dust',mode,0);p.add(4e-7,1e-6,'old')
        p.reconcile_solver_opening(0,'next')
        assert p.mass_kg==0 and p.carbon_kgco2==0
        assert abs(p.reconciliation_carbon_change+1e-6)<1e-15
        try:
            p.reconcile_solver_opening(1e-4,'untraceable')
        except ValueError: pass
        else: raise AssertionError('Untraceable empty mass accepted')
        a=CohortPool('g',mode,10);b=CohortPool('ng',mode,0)
        b.add(10,30,'new');assert a.withdraw(5,'now')==0
        assert b.withdraw(5,'now')==15
        records.append({'test':f'{mode}: identical CI, partial draw, dust sink, empty rejection, label independence','status':'PASS','details':'Analytical expectations'})
    # Independent 4-hour process fixture: first create carbon-bearing H2/NH3,
    # then synthesis + cracking + fuel, then sell. Expected carbon is derived
    # here directly, without calling the allocation helpers to form expectations.
    stocks={n:0.0 for n in POOL_NAMES}; rows=[]
    for i in range(4):
        r={c:0.0 for c in REQUIRED_MODEL_COLS}
        r.update(model_tag='analytical_process',datetime_utc=pd.Timestamp('2024-01-01',tz='UTC')+pd.Timedelta(hours=i),day_of_year=1,hour_in_day=i)
        for n in POOL_NAMES:r[f'tank_{n}_kg']=stocks[n]
        if i==0:r.update(H2_prod_ng_kgph=100.,NH3_prod_ng_kgph=0.,P_elz_mw=.5)
        if i==1:r.update(NH3_prod_ng_kgph=10.,MeOH_prod_ng_kgph=10.,P_nh3_mw=.01,P_meoh_mw=.02)
        if i==2:r.update(crk_nh3_in_kgph=2.,crk_h2_out_kgph=2*K_H2_OUT,crk_h2_fuel_kgph=2*K_H2_FUEL)
        if i==3:r.update(MeOH_sell_ng_kgph=5.,NH3_sell_ng_kgph=1.)
        rows.append(r)
        q=2 if i==2 else 0
        stocks['H2_ng']+=r['H2_prod_ng_kgph']-M_H2_TO_NH3*r['NH3_prod_ng_kgph']-M_H2_TO_MEOH*r['MeOH_prod_ng_kgph']+(K_H2_OUT-K_H2_FUEL)*q
        stocks['NH3_ng']+=r['NH3_prod_ng_kgph']-r['NH3_sell_ng_kgph']-(1-K_NH3_REC)*q
        stocks['MeOH_ng']+=r['MeOH_prod_ng_kgph']-r['MeOH_sell_ng_kgph']
    d,_=reconstruct_unit_grid_battery_split(pd.DataFrame(rows));d,_=reconstruct_intermediates(d)
    sig=pd.DataFrame({'datetime_utc':d.datetime_utc,'ef_kg_per_kwh':.2})
    for mode in METHODS:
        h,a,*_=run_fifo_lifo_ledger(d,'test',sig,mode)
        nh=1.8+10*DE_NH3+2;me=2.2+10*DE_MEOH+4
        assert abs(h.loc[1,'C_added_NH3_ng_kgco2']-nh)<1e-10
        assert abs(h.loc[1,'C_added_MeOH_ng_kgco2']-me)<1e-10
        assert abs(h.loc[2,'C_added_H2_ng_kgco2']-(nh/10*2*(1-K_NH3_REC)))<1e-10
        assert abs(a.loc[2,'cracker_fuel_carbon_sink_kgco2']-2*K_H2_FUEL)<1e-10
        assert abs(h.loc[3,'C_sell_MeOH_ng_kgco2']-me/2)<1e-10
        assert h.loc[0,'MeOH_sold_CI_kgco2_perkg_hourly']!=h.loc[0,'MeOH_sold_CI_kgco2_perkg_hourly']
    records.append({'test':'Coupled process with nonzero inherited/grid/direct carbon, return, recovery and fuel','status':'PASS','details':'Independent analytical carbon expectations'})
    return pd.DataFrame(records)


def verify_real_outputs(raw, hourly, audit, pool):
    """Independent output checks and telescoping carbon closure."""
    if len(hourly)!=8784 or not hourly.datetime_utc.equals(raw.datetime_utc):
        raise AssertionError('Real-data hour coverage/order changed')
    for p in PRODUCTS:
        for lab in LABELS:
            source=raw[f'{p}_sell_{lab}_kgph'].clip(lower=0).to_numpy()
            if not np.array_equal(source,hourly[f'{p}_sold_{lab}_kgph'].to_numpy()):
                raise AssertionError(f'Sales changed: {p}_{lab}')
        m=hourly[f'{p}_sold_total_kgph']; c=hourly[f'C_sell_{p}_g_kgco2']+hourly[f'C_sell_{p}_ng_kgco2']
        ci=hourly[f'{p}_sold_CI_kgco2_perkg_hourly']; good=m>0
        if not np.isfinite(ci[good]).all() or not ci[~good].isna().all():
            raise AssertionError(f'Invalid positive/zero-sales CI: {p}')
        if not np.allclose(ci[good]*m[good],c[good],rtol=1e-12,atol=1e-8):
            raise AssertionError('Incorrect CI ratio')
    # Final state is an accounting-only endpoint, explicitly not validated mass.
    initial=audit.opening_carbon_stock_kgco2.iloc[0]
    final=pool.loc[pool.datetime_utc==hourly.datetime_utc.iloc[-1],'cohort_after_hour_carbon_kgco2'].sum()
    net=(audit.new_electricity_carbon_kgco2+audit.new_direct_emission_kgco2
         -audit.sold_product_carbon_kgco2-audit.cracker_fuel_carbon_sink_kgco2
         -audit.unassigned_cracker_electricity_kgco2-audit.excluded_numerical_carbon_residual_kgco2
         +audit.carbon_reconciliation_change_kgco2).sum()
    residual=final-initial-net
    if abs(residual)>1e-4: raise AssertionError(f'Cumulative carbon closure failed: {residual}')
    return residual



def main():
    print('Running FIFO/LIFO scientific unit tests...')
    global RECONCILIATION_POLICY
    policy=RECONCILIATION_POLICY
    RECONCILIATION_POLICY='preserve_carbon'
    tests=pd.concat([self_tests(),extra_tests()],ignore_index=True)
    RECONCILIATION_POLICY=policy
    print(tests[['test','status']].to_string(index=False))
    signal_data={}
    for signal in EVALUATION_SIGNALS:
        if signal not in SIGNAL_FILES:
            raise KeyError(f'Unknown signal: {signal}')
        filename,col=SIGNAL_FILES[signal]
        signal_data[signal]=load_signal_df(BASE_DIR/filename,col)
    models={name:BASE_DIR/f for name,f in MODEL_FILES.items()}
    missing=[f'{name}: {f}' for name,f in models.items() if not f.exists()]
    if missing:
        raise FileNotFoundError('Missing requested models. Remove/comment optional MODEL_FILES entries '
                                'or upload files:\n'+'\n'.join(missing))
    if 'M0' not in models:
        raise ValueError('M0 is needed for paired M2-vs-M0 comparisons')
    timestamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    root=OUTPUT_DIR/f'fifo_lifo_v8_{timestamp}'
    if root.exists():
        raise FileExistsError(f'Output already exists, aborting: {root}')
    (root/'tables').mkdir(parents=True)
    (root/'audit').mkdir()
    (root/'per_model').mkdir()
    tests.to_csv(root/'audit'/'synthetic_unit_tests.csv',index=False)
    pd.DataFrame([{'filename':f.name,'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in list(models.values())+[BASE_DIR/SIGNAL_FILES[x][0] for x in EVALUATION_SIGNALS]]).to_csv(root/'audit'/'input_sha256.csv',index=False)
    config=pd.DataFrame([
        {'item':'version','value':'V8 2026-11-03'},
        {'item':'reconciliation_policy','value':RECONCILIATION_POLICY},
        {'item':'methods','value':', '.join(METHODS)},
        {'item':'signals','value':', '.join(EVALUATION_SIGNALS)},
        {'item':'models','value':', '.join(models)},
        {'item':'cohort_definition','value':'One aggregate cohort per hour / product / label'},
        {'item':'virtual_labels','value':'g and ng remain distinct; no cross-label carbon reassignment'},
        {'item':'same_hour_withdrawals','value':'Total withdrawal FIFO/LIFO, carbon proportional to fixed destination mass'},
        {'item':'within_hour_timing','value':'Opening stock consumed; all new cohorts enter at hour end'},
        {'item':'opening_inventory','value':'Solver opening mass, zero assigned carbon burden'},
        {'item':'last_hour','value':'Original cracker feed split unavailable; no validated final closing carbon stocks'},
        {'item':'reference','value':'Author original ex-post code; NO perfect-mixing recalculation'},
        {'item':'zero_mass_grid_carbon_limit_kgco2',
         'value':str(ZERO_MASS_GRID_CARBON_RESIDUAL_LIMIT_KGCO2)},
        {'item':'zero_mass_arithmetic_roundoff_limit_kgco2',
         'value':str(ZERO_MASS_ARITHMETIC_ROUNDOFF_LIMIT_KGCO2)},
        {'item':'signed_near_zero_source_material','value':
         'Uniform 0.001 kg/h negative-mass clipping at ledger input; same cleaned production for material and direct emissions; audit each correction'},
        {'item':'mass_reconciliation','value':
         'Proportional cohort-mass scaling; min(0.010 kg, 0.001 kg + 1e-5 * max(1, total physical-product opening kg)); empty-state dust <=1e-5 kg and <=0.001 kgCO2 is excluded and logged; policy separately recorded'},
        {'item':'preflight','value':'All six material balances checked hourly using source flows; refuse cases above reconciliation limits'},
        {'item':'numerical_residual_handling','value':
         'Carbon-only additions excluded if <=1e-8 kgCO2 arithmetic roundoff, or if <=0.001 kgCO2 and matching non-green source-grid electricity; explicitly logged plant-carbon sink'},
    ])
    config.to_csv(root/'audit'/'accounting_method_assumptions.csv',index=False)
    all_hourly,annual_rows,all_audit,all_pool,all_final,all_diag,all_traces,all_residuals,all_mass_corrections,all_mass_reconciliations=[],[],[],[],[],[],[],[],[],[]
    for model,source in models.items():
        print(f'Loading saved optimizer flows: {model}')
        raw=load_model_df(source)
        share_events=[]
        for unit in ('elz','nh3','meoh','crk'):
            col=f'{unit}_power_green_share'
            affected=raw[col].isna() | (raw[col]<0) | (raw[col]>1)
            for _,r in raw.loc[affected].iterrows():
                share_events.append({'datetime_utc':r.datetime_utc,'unit':unit,
                    'raw_share':r[col],'used_share':float(np.clip(r[col],0,1)) if pd.notna(r[col]) else 0.0,
                    'unit_power_mw':r[f'P_{unit}_mw'],'reason':'Original idle-unit fillna/clip convention'})
        pd.DataFrame(share_events).to_csv(root/'audit'/f'power_share_normalization_{model[:2]}.csv',index=False)
        reconstructed,power_diag=reconstruct_unit_grid_battery_split(raw)
        reconstructed,cracker_diag=reconstruct_intermediates(reconstructed)
        # Validate source material balances before any FIFO/LIFO calculations.
        print(f'  Preflight all six source material balances: {model}')
        stages=source_stage_audit(raw,reconstructed)
        stages.to_csv(root/'audit'/f'source_stages_{model[:2]}.csv',index=False)
        stage_summary(stages).to_csv(root/'audit'/f'source_stage_summary_{model[:2]}.csv',index=False)
        focus=stages.datetime_utc.between(pd.Timestamp('2024-06-17 00:00',tz='UTC'),pd.Timestamp('2024-06-17 03:00',tz='UTC')) if model=='M0' else stages.datetime_utc.between(pd.Timestamp('2024-06-24 00:00',tz='UTC'),pd.Timestamp('2024-06-24 10:00',tz='UTC'))
        stages.loc[focus].to_csv(root/'audit'/f'regression_rows_{model[:2]}.csv',index=False)
        source_mass_audit=material_source_balance_audit(reconstructed)
        source_mass_audit.insert(0,'Model',model)
        source_mass_audit.to_csv(root/'audit'/f'source_material_balance_{model.replace(" ","_").replace("/","_")}.csv',index=False)
        for diagnostic,table in [('power',power_diag),('cracker',cracker_diag)]:
            x=table.copy();x.insert(0,'Model',model);x.insert(1,'block',diagnostic)
            all_diag.append(x)
        model_dir=root/'per_model'/model.replace(' ','_').replace('/','_')
        model_dir.mkdir()
        for signal,signal_df in signal_data.items():
            for mode in METHODS:
                print(f'  {model} | {signal} | {mode}')
                (hourly,audit,pool,final_counts,trace,residuals,mass_corrections,
                 mass_reconciliations)=run_fifo_lifo_ledger(reconstructed, signal, signal_df, mode)
                cumulative_gap=verify_real_outputs(raw,hourly,audit,pool)
                audit['cumulative_accounting_carbon_gap_kgco2']=cumulative_gap
                hourly.insert(0,'Model',model)
                audit.insert(0,'Model',model)
                pool.insert(0,'Model',model)
                final_counts.insert(0,'Model',model)
                daily=summarize_daily(hourly)
                daily.insert(0,'Model',model)
                annual=summarize_annual(hourly,model)
                annual_rows.append(annual)
                tag=signal.lower().replace(' ','_')+'_'+mode.lower()
                hourly.to_csv(model_dir/f'03_hourly_carbon_ledger_{tag}.csv',index=False)
                daily.to_csv(model_dir/f'04_daily_product_CI_{tag}.csv',index=False)
                pd.DataFrame([annual]).to_csv(model_dir/f'05_annual_product_CI_{tag}.csv',index=False)
                all_hourly.append(hourly)
                all_audit.append(audit)
                all_pool.append(pool)
                all_final.append(final_counts)
                trace.insert(0,'Accounting method',mode)
                trace.insert(0,'Evaluation signal',signal)
                trace.insert(0,'Model',model)
                all_traces.append(trace)
                residuals.insert(0,'Model',model)
                all_residuals.append(residuals)
                mass_corrections.insert(0,'Model',model)
                all_mass_corrections.append(mass_corrections)
                mass_reconciliations.insert(0,'Model',model)
                all_mass_reconciliations.append(mass_reconciliations)
    annual=pd.DataFrame(annual_rows)
    joined_hourly=pd.concat(all_hourly,ignore_index=True)
    joined_audit=pd.concat(all_audit,ignore_index=True)
    joined_pool=pd.concat(all_pool,ignore_index=True)
    annual.to_csv(root/'tables'/'all_models_all_methods_annual_product_CI.csv',index=False)
    paired_annual(annual).to_csv(root/'tables'/'annual_reduction_vs_M0_by_method.csv',index=False)
    hourly_distribution(joined_hourly).to_csv(root/'tables'/'hourly_CI_distribution_by_method.csv',index=False)
    paired_hourly(joined_hourly).to_csv(root/'tables'/'paired_hourly_M2_vs_M0_by_method.csv',index=False)
    joined_audit.to_csv(root/'audit'/'hourly_plant_carbon_conservation.csv',index=False)
    joined_pool.to_csv(root/'audit'/'hourly_six_pool_mass_checks.csv',index=False)
    pd.concat(all_final,ignore_index=True).to_csv(root/'audit'/'unverified_last_hour_cohort_status.csv',index=False)
    pd.concat(all_traces,ignore_index=True).to_csv(root/'audit'/'cohort_withdrawal_trace.csv',index=False)
    pd.concat(all_diag,ignore_index=True).to_csv(root/'audit'/'original_reconstruction_diagnostics.csv',index=False)
    pd.concat(all_residuals,ignore_index=True).to_csv(
        root/'audit'/'numerical_carbon_residual_events.csv',index=False)
    pd.concat(all_mass_corrections,ignore_index=True).to_csv(
        root/'audit'/'numerical_mass_flow_corrections.csv',index=False)
    joined_reconciliations=pd.concat(all_mass_reconciliations,ignore_index=True)
    joined_reconciliations.to_csv(root/'audit'/'cohort_mass_reconciliation_events.csv',index=False)
    summary=[]
    for (model,mode),g in joined_audit.groupby(['Model','Accounting method'],sort=True):
        summary.append({
            'Model':model,'Accounting method':mode,
            'max_abs_opening_mass_gap_kg':float(g.pool_opening_max_abs_mass_gap_kg.max()),
            'max_abs_gap_before_reconciliation_kg':float(g.pool_opening_max_abs_gap_before_reconciliation_kg.max()),
            'n_cohort_mass_reconciliation_events':int(((joined_reconciliations.Model==model)&(joined_reconciliations['Accounting method']==mode)).sum()),
            'total_abs_cohort_mass_reconciliation_kg':float(joined_reconciliations.loc[(joined_reconciliations.Model==model)&(joined_reconciliations['Accounting method']==mode),'absolute_mass_adjustment_kg'].sum()),
            'max_abs_closing_mass_gap_kg':float(g.pool_closing_max_abs_mass_gap_kg.max()),
            'max_abs_carbon_balance_residual_kgco2':float(g.plant_carbon_balance_residual_kgco2.abs().max()),
            'n_hours_with_unavailable_cracker_split':int((~g.cracker_feed_split_valid).sum()),
            'n_hours_with_reconstructed_small_negative_flows':int((g.corrected_small_negative_cracker_feed_kg>0).sum()),
            'total_corrected_small_negative_flow_kg':float(g.corrected_small_negative_cracker_feed_kg.sum()),
            'all_source_mass_negative_roundoff_kg':float(g.corrected_small_negative_all_source_flows_kg.sum()),
            'all_source_mass_negative_roundoff_events':int(g.n_corrected_small_negative_all_source_flows.sum()),
            'total_cohort_withdrawal_roundoff_kg':float(g.cohort_withdrawal_roundoff_kg_cumulative.iloc[-1]),
            'total_excluded_numerical_carbon_residual_kgco2':float(g.excluded_numerical_carbon_residual_kgco2.sum()),
            'n_hours_with_excluded_numerical_carbon':int((g.excluded_numerical_carbon_residual_kgco2>0).sum()),
            'total_net_cohort_mass_reconciliation_kg':float(joined_reconciliations.loc[(joined_reconciliations.Model==model)&(joined_reconciliations['Accounting method']==mode),'signed_mass_adjustment_kg'].sum()),
            'carbon_reconciliation_change_net_kgco2':float(g.carbon_reconciliation_change_kgco2.sum()),
            'carbon_reconciliation_change_abs_kgco2':float(g.carbon_reconciliation_change_kgco2.abs().sum()),
            'cumulative_accounting_carbon_gap_kgco2':float(g.cumulative_accounting_carbon_gap_kgco2.iloc[0]),
            'final_hour_closing_stocks_verified':False,
            'audit_status':'PASS (validated nonfinal openings/closings and carbon conservation)',
        })
    pd.DataFrame(summary).to_csv(root/'audit'/'audit_summary.csv',index=False)
    (root/'README.txt').write_text('V8 FIFO/LIFO attribution sensitivity. Read audit_summary.csv, source_stage_summary files and accounting_method_assumptions.csv before using results. Last-hour closing inventory is unverified. Negative source flows are normalized and audited; raw input files are unchanged. Source-stage NH3 raw closure is by construction. Generated '+timestamp+'; policy '+RECONCILIATION_POLICY+'\n',encoding='utf-8')
    zip_path=OUTPUT_DIR/(root.name+'.zip')
    if zip_path.exists():
        raise FileExistsError(f'Archive already exists: {zip_path}')
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6) as z:
        for file in sorted(root.rglob('*')):
            if not file.is_file(): continue
            z.write(file,arcname=str(file.relative_to(root)))
    print('\nAll tests and required scientific audits passed.')
    print('Results directory:',root)
    print('CSV ZIP:',zip_path)
    print('Use audit/audit_summary.csv and audit/cohort_mass_reconciliation_events.csv,')
    print('audit/source_material_balance_<model>.csv, and audit/synthetic_unit_tests.csv before interpreting results.')
    return zip_path


if __name__ == '__main__':
    main()


Running FIFO/LIFO scientific unit tests...
                                                                                test status
                FIFO/LIFO distinct chronological withdrawals and carbon preservation   PASS
                      Simultaneous process/sale withdrawals allocated proportionally   PASS
                              No reset at midnight; FIFO carries cohorts across days   PASS
                                       Incoming material unavailable until next hour   PASS
                     Ammonia-cracker carbon inheritance and electricity conservation   PASS
            Traceable 0.001 kgCO2 carbon-only grid residual; reject unrelated carbon   PASS
   Green-pool ~1.4e-9 kgCO2 roundoff: audited suppression, stricter error above 1e-8   PASS
      V5: preserve small real cohort-carbon stocks over repeated partial withdrawals   PASS
             End-to-end zero-product grid residual: event log and exact conservation   PASS
                       V6: signed nea